<!-- Curated copy -->
> **Curated copy.** This notebook is taken verbatim from the BTech-thesis working archive; only
> cell *outputs* have been cleared and machine-specific absolute paths (`C:\\...`, `D:\\...`,
> `F:\\...`) have been rewritten to repository-relative `runs/...` paths. No scientific logic,
> equation, hyper-parameter or architecture has been modified. Place regenerated
> `dataset_run_*` folders under a `runs/` directory next to this notebook (or edit the paths).
> The figures this notebook originally produced are preserved in the sibling `figures/` folder.


In [ ]:
import os, json,ast, math, glob, time
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
import glob
import pandas as pd
from sklearn.decomposition import PCA

In [ ]:
class HeatSource2DRBFPosterior:
    """
    Generate 2D heat-source fields Q(x,y) via a Gaussian-process posterior
    with an RBF (squared-exponential) kernel on a square grid.
    """

    def __init__(self, grid_size=64, length=1.0,
                 length_scale=0.20, sigma=1.0,
                 jitter=1e-6, seed=None):
        self.N = int(grid_size)
        self.L = float(length)
        self.l = float(length_scale)
        self.sigma = float(sigma)
        self.jitter = float(jitter)
        self.rng = np.random.default_rng(seed)

        x = np.linspace(0.0, self.L, self.N, dtype=np.float64)
        y = np.linspace(0.0, self.L, self.N, dtype=np.float64)
        self.X, self.Y = np.meshgrid(x, y, indexing="ij")
        self.points = np.column_stack((self.X.ravel(), self.Y.ravel()))

        # Boolean mask of boundary grid points
        self.boundary_mask = (
            np.isclose(self.X, 0.0) | np.isclose(self.X, self.L) |
            np.isclose(self.Y, 0.0) | np.isclose(self.Y, self.L)
        ).ravel()

    # RBF (Gaussian) kernel
    def rbf_kernel(self, X1, X2):
        d2 = np.sum((X1[:, None, :] - X2[None, :, :])**2, axis=-1)
        return (self.sigma**2) * np.exp(-0.5 * d2 / (self.l**2))

    # Robust Cholesky with adaptive jitter
    def safe_cholesky(self, K, max_tries=8):
        jitter = self.jitter
        I = np.eye(K.shape[0], dtype=K.dtype)
        for _ in range(max_tries):
            try:
                return np.linalg.cholesky(K + jitter * I)
            except np.linalg.LinAlgError:
                jitter *= 10.0
        raise np.linalg.LinAlgError(f"Cholesky failed; final jitter tried={jitter:g}")

    @staticmethod
    def _softplus(x, beta=6.0):
        # smooth nonnegative map; larger beta -> closer to max(0,x)
        return (1.0/beta) * np.log1p(np.exp(beta*x))

    def sample_posterior(
        self,
        Qb_func=None,
        enforce_zero_mean=True,
        *,
        force_sign=None,            # None | 'positive' | 'negative'
        pos_beta=6.0,               # sharpness for softplus
        target_mean=None,           # set desired mean after transform (e.g., 5e4)
        keep_boundary_zero=True     # keep Q=0 on the boundary after transform
    ):
        """
        Draw a sample Q from the GP posterior conditioned on boundary values Q_b.

        Parameters
        ----------
        Qb_func : callable or None
            Function returning boundary values Q_b at boundary coordinates;
            if None, boundary is conditioned to Q=0 in distribution.
        enforce_zero_mean : bool
            If True and force_sign is None, recenter to zero mean.
        force_sign : None | 'positive' | 'negative'
            Enforce all-positive or all-negative field using a smooth transform.
        pos_beta : float
            Softplus sharpness for positivity; 5–8 is typical.
        target_mean : float or None
            If given, shift the final field so mean(Q) == target_mean.
        keep_boundary_zero : bool
            If True, zero out the boundary *after* any sign/mean transforms.

        Returns
        -------
        Q : (N,N) ndarray
        """
        X_all = self.points
        X_b = X_all[self.boundary_mask]

        # Boundary values for Q
        if Qb_func is None:
            Q_b = np.zeros(len(X_b), dtype=np.float64)
        else:
            Q_b = np.asarray(Qb_func(X_b), dtype=np.float64)

        # Posterior mean and covariance
        K_xx = self.rbf_kernel(X_all, X_all).astype(np.float64)
        K_xb = self.rbf_kernel(X_all, X_b).astype(np.float64)
        K_bb = self.rbf_kernel(X_b, X_b).astype(np.float64) + self.jitter * np.eye(len(X_b))

        mu = K_xb @ np.linalg.solve(K_bb, Q_b)
        K_post = K_xx - K_xb @ np.linalg.solve(K_bb, K_xb.T)

        # Sample from posterior (Gaussian)
        Lp = self.safe_cholesky(K_post)
        z = self.rng.standard_normal(self.N * self.N)
        Q = (mu + Lp @ z).reshape(self.N, self.N)

        # Enforce sign if requested
        if force_sign is None:
            if enforce_zero_mean:
                Q -= Q.mean()
        elif force_sign == 'positive':
            Q = self._softplus(Q, beta=pos_beta)  # strictly >= 0
        elif force_sign == 'negative':
            Q = -self._softplus(Q, beta=pos_beta) # strictly <= 0
        else:
            raise ValueError("force_sign must be None, 'positive', or 'negative'")

        # Optional mean targeting *after* transform
        if target_mean is not None:
            Q += (target_mean - Q.mean())

        # Optionally keep boundary exactly zero (useful for coupling to PDE BCs)
        if keep_boundary_zero:
            Q.ravel()[self.boundary_mask] = 0.0

        return Q

    @staticmethod
    def show(Q, title="Heat source Q(x,y)", cmap="RdBu_r"):
        plt.figure(figsize=(5.0, 4.5), dpi=120)
        im = plt.imshow(Q.T, origin="lower", cmap=cmap, aspect="equal")
        plt.colorbar(im, shrink=0.9, label="Q")
        plt.title(title)
        plt.xlabel("x index")
        plt.ylabel("y index")
        plt.tight_layout()
        plt.show()


In [ ]:
#example
if __name__ == "__main__":
    gen = HeatSource2DRBFPosterior(
        grid_size=64, length=1.0, length_scale=0.02, sigma=0.65,       # length = 1.0
        jitter=1e-6, seed=7
    )

    #Q=0 on the boundary (default)
    Q0 = gen.sample_posterior(Qb_func=None, enforce_zero_mean=True)
    gen.show(Q0, title="RBF‑GP heat source (Q=0 on boundary)")

    #Heterogeneous boundary for Q (e.g., sinusoidal sides)
    def Qb(xb):
        x, y = xb[:, 0], xb[:, 1]
        return 2.0*np.sin(2*np.pi*x)*(y*(1.0-y)) + 1.5*np.sin(2*np.pi*y)*(x*(1.0-x))

    Q1 = gen.sample_posterior(Qb_func=Qb, enforce_zero_mean=True)
    gen.show(Q1, title="RBF‑GP heat source (heterogeneous)")

In [ ]:
def _const_profile(val):
    """returns a function f(s) -> constant temperature array"""
    return lambda s: np.full_like(s, float(val), dtype=float)

def _gauss_profile(base, amp, mu, sigma, axis_len):
    """
    1D Gaussian along coordinate s in [0, axis_len]:
    T(s) = base + amp * exp(-0.5 * ((s - mu*axis_len)/(sigma*axis_len))**2)
    mu and sigma are given in 0..1 (relative position / width).
    """
    mu_abs = mu * axis_len
    sig_abs = max(1e-12, sigma * axis_len)
    return lambda s: base + amp * np.exp(-0.5 * ((s - mu_abs)/sig_abs)**2)

def _make_profile(side_cfg, axis_array, axis_len):
    """
    side_cfg: dict like {"on": True/False, "type": "const"/"gauss"/"custom", **params}
    axis_array: x (for top/bottom) or y (for left/right)
    axis_len: Lx (for top/bottom) or Ly (for left/right)
    returns: (on_bool, array_of_temperatures) or (False, None) if OFF
    """
    on = bool(side_cfg.get("on", False))
    if not on:
        return False, None

    typ = side_cfg.get("type", "const").lower()
    if typ == "const":
        prof = _const_profile(side_cfg.get("T", 300.0))
    elif typ == "gauss":
        prof = _gauss_profile(
            base=float(side_cfg.get("base", 330.0)),
            amp=float(side_cfg.get("amp", 20.0)),
            mu=float(side_cfg.get("mu", 0.5)),
            sigma=float(side_cfg.get("sigma", 0.2)),
            axis_len=axis_len,
        )
    elif typ == "custom":
        func = side_cfg.get("func", None)
        if not callable(func):
            raise ValueError("bc 'custom' requires a callable 'func(s_array) -> T_array'")
        prof = func
    else:
        raise ValueError(f"Unknown bc type: {typ}")

    return True, prof(axis_array)

In [ ]:
def simulate_pcm_2d_with_source(
    Q_Wm3,
    nx=128, ny=128, Lx=0.05, Ly=0.05,
    rho=800.0, cp=2000.0, k=0.2, L_lat=2e5, Tm=330.0,
    T_init=300.0, Tb=300.0,
    t_end=2500.0, cfl=0.45,
    save_times=(10.0, 60.0, 300.0, 600.0, 1200.0, 2000.0, 2500.0),
    bc=None,
    battery_mask=None,
):
    """
    Q_Wm3: [nx,ny] in (x,y) indexing; x is axis-0, y is axis-1.
    battery_mask: [nx,ny] with 1 in battery region.
      - We enforce: f=0 in battery region (no melting there).
      - Temperature still evolves by conduction/enthalpy there (solid-like region).
    """

    # grid
    x = np.linspace(0.0, Lx, nx)
    y = np.linspace(0.0, Ly, ny)
    dx = x[1] - x[0]
    dy = y[1] - y[0]
    X, Y = np.meshgrid(x, y, indexing="ij")  # [nx,ny]

    # fields
    T = np.full((nx, ny), float(T_init), dtype=np.float64)
    f = np.zeros_like(T)
    H = cp*T + f*L_lat

    # default BCs (same behavior as your notebook if bc None)
    if bc is None:
        bc = {
            "left":   {"on": True, "type": "const", "T": Tb},
            "right":  {"on": True, "type": "const", "T": Tb},
            "bottom": {"on": True, "type": "const", "T": Tb},
            "top":    {"on": True, "type": "const", "T": Tb},
        }

    # battery mask
    if battery_mask is not None:
        battery_mask = (np.asarray(battery_mask) > 0.5)
    else:
        battery_mask = None

    # axis arrays for boundary profiles
    xs = X[:, 0]   # along bottom/top edges (x coordinate)
    ys = Y[0, :]   # along left/right edges (y coordinate)

    # timestep (2D FTCS stability-ish)
    alpha = k/(rho*cp)
    dt = cfl * min(dx*dx, dy*dy) / (4.0*alpha)

    save_times = np.array(sorted(save_times), dtype=float)
    Ts, Fs, ts = [], [], []
    next_k = 0
    t = 0.0

    def apply_BCs(Tarr, Harr):
        # LEFT
        on, arr = _make_profile(bc.get("left", {}), ys, Ly)
        if on:
            Tarr[0, :] = arr
            Harr[0, :] = cp*Tarr[0, :] + f[0, :]*L_lat
        else:
            Tarr[0, :] = Tarr[1, :]  # adiabatic

        # RIGHT
        on, arr = _make_profile(bc.get("right", {}), ys, Ly)
        if on:
            Tarr[-1, :] = arr
            Harr[-1, :] = cp*Tarr[-1, :] + f[-1, :]*L_lat
        else:
            Tarr[-1, :] = Tarr[-2, :]

        # BOTTOM
        on, arr = _make_profile(bc.get("bottom", {}), xs, Lx)
        if on:
            Tarr[:, 0] = arr
            Harr[:, 0] = cp*Tarr[:, 0] + f[:, 0]*L_lat
        else:
            Tarr[:, 0] = Tarr[:, 1]

        # TOP
        on, arr = _make_profile(bc.get("top", {}), xs, Lx)
        if on:
            Tarr[:, -1] = arr
            Harr[:, -1] = cp*Tarr[:, -1] + f[:, -1]*L_lat
        else:
            Tarr[:, -1] = Tarr[:, -2]

    # initial BC
    apply_BCs(T, H)

    while t < t_end + 1e-12:
        apply_BCs(T, H)

        # Laplacian (interior)
        Lap = np.zeros_like(T)
        Lap[1:-1, 1:-1] = (
            (T[2:, 1:-1] - 2*T[1:-1, 1:-1] + T[:-2, 1:-1]) / dx**2 +
            (T[1:-1, 2:]   - 2*T[1:-1, 1:-1] + T[1:-1, :-2]) / dy**2
        )

        # Enthalpy update (interior only)
        H[1:-1, 1:-1] += dt * ((k/rho) * Lap[1:-1, 1:-1] + Q_Wm3[1:-1, 1:-1] / rho)

        # Phase update:
        # - normal PCM everywhere, BUT force f=0 in battery cells (no melt inside batteries)
        if battery_mask is None:
            f_new = (H - cp*Tm) / L_lat
            mush  = (f_new > 0.0) & (f_new < 1.0)
            solid = (f_new <= 0.0)
            liq   = (f_new >= 1.0)

            T[mush]  = Tm
            f[mush]  = f_new[mush]
            T[solid] = H[solid]/cp
            f[solid] = 0.0
            T[liq]   = (H[liq]-L_lat)/cp
            f[liq]   = 1.0
        else:
            pcm = ~battery_mask

            # update only PCM region as phase change material
            f_new_pcm = (H[pcm] - cp*Tm) / L_lat
            mush  = (f_new_pcm > 0.0) & (f_new_pcm < 1.0)
            solid = (f_new_pcm <= 0.0)
            liq   = (f_new_pcm >= 1.0)

            # defaults in PCM: solid relation
            T[pcm] = H[pcm]/cp
            f[pcm] = 0.0

            pcm_idx = np.flatnonzero(pcm.ravel())

            mush_idx = pcm_idx[mush]
            liq_idx  = pcm_idx[liq]

            # mush
            T.ravel()[mush_idx] = Tm
            f.ravel()[mush_idx] = f_new_pcm[mush]

            # liquid
            T.ravel()[liq_idx] = (H.ravel()[liq_idx] - L_lat)/cp
            f.ravel()[liq_idx] = 1.0

            # battery region (no melt)
            T[battery_mask] = H[battery_mask]/cp
            f[battery_mask] = 0.0

        apply_BCs(T, H)

        # Save snapshots at requested times
        if next_k < len(save_times) and t >= save_times[next_k] - 0.5*dt:
            Ts.append(T.copy())
            Fs.append(f.copy())
            ts.append(t)
            next_k += 1

        t += dt

    return np.array(Ts), np.array(Fs), np.array(ts), X, Y, {"dt": dt}

In [ ]:
#def simulate_pcm_2d_with_source(
#    Q_Wm3,
#    nx=128, ny=128, Lx=0.05, Ly=0.05,
#    rho=800.0, cp=2000.0, k=0.2, L_lat=2e5, Tm=330.0,
#    T_init=300.0, Tb=300.0,
#    t_end=1200.0, cfl=0.45, save_times=(60.0, 300.0, 600.0, 1200.0),
#    # ---- new: boundary configuration ----
#    bc=None,
#):
#    """
#    bc (dict) controls each side:
#      bc = {
#        "left":   {"on": True,  "type": "const", "T": 350.0},
#        "right":  {"on": False},                                   # OFF -> adiabatic
#        "bottom": {"on": True,  "type": "gauss", "base":330, "amp":20, "mu":0.5, "sigma":0.2},
#        "top":    {"on": True,  "type": "custom", "func": lambda s: 335+10*np.sin(2*np.pi*s/Lx)},
#      }
#    If bc is None, the old behavior is kept: all four sides ON at Tb (constant).
#    """
#
#    # grid
#    x = np.linspace(0.0, Lx, nx); y = np.linspace(0.0, Ly, ny)
#    dx = x[1] - x[0]; dy = y[1] - y[0]
#    X, Y = np.meshgrid(x, y, indexing="ij")
#
#    # fields
#    T = np.full((nx, ny), T_init, dtype=np.float64)
#    f = np.zeros_like(T)
#    H = cp*T + f*L_lat
#
#    # default BCs = your previous "all Dirichlet at Tb"
#    if bc is None:
#        bc = {
#            "left":   {"on": True,  "type": "const", "T": Tb},
#            "right":  {"on": True,  "type": "const", "T": Tb},
#            "bottom": {"on": True,  "type": "const", "T": Tb},
#            "top":    {"on": True,  "type": "const", "T": Tb},
#        }
#
#    # precompute axis arrays for profiles
#    xs = X[:, 0]        # along bottom/top edges
#    ys = Y[0, :]        # along left/right edges
#
#    # explicit stability bound for 2D FTCS
#    alpha = k/(rho*cp)
#    dt = cfl * min(dx*dx, dy*dy) / (4.0*alpha)
#
#    save_times = np.array(sorted(save_times), dtype=float)
#    Ts, Fs, ts, next_k = [], [], [], 0
#    t = 0.0
#
#    def apply_BCs(Tarr, Harr):
#        """
#        Apply per-side boundary modes:
#          - ON  -> Dirichlet with given profile
#          - OFF -> adiabatic (copy interior)
#        Re-sync H on Dirichlet sides.
#        """
#        # LEFT
#        on, arr = _make_profile(bc.get("left", {}), ys, Ly)
#        if on:
#            Tarr[0, :] = arr
#            Harr[0, :] = cp*Tarr[0, :] + f[0, :]*L_lat
#        else:
#            Tarr[0, :] = Tarr[1, :]  # adiabatic
#
#        # RIGHT
#        on, arr = _make_profile(bc.get("right", {}), ys, Ly)
#        if on:
#            Tarr[-1, :] = arr
#            Harr[-1, :] = cp*Tarr[-1, :] + f[-1, :]*L_lat
#        else:
#            Tarr[-1, :] = Tarr[-2, :]
#
#        # BOTTOM
#        on, arr = _make_profile(bc.get("bottom", {}), xs, Lx)
#        if on:
#            Tarr[:, 0] = arr
#            Harr[:, 0] = cp*Tarr[:, 0] + f[:, 0]*L_lat
#        else:
#            Tarr[:, 0] = Tarr[:, 1]
#
#        # TOP
#        on, arr = _make_profile(bc.get("top", {}), xs, Lx)
#        if on:
#            Tarr[:, -1] = arr
#            Harr[:, -1] = cp*Tarr[:, -1] + f[:, -1]*L_lat
#        else:
#            Tarr[:, -1] = Tarr[:, -2]
#
#    # initial boundary imposition
#    apply_BCs(T, H)
#
#    while t < t_end + 1e-12:
#        # enforce BCs before building Laplacian (safety)
#        apply_BCs(T, H)
#
#        # five-point Laplacian (interior only)
#        Lap = np.zeros_like(T)
#        Lap[1:-1,1:-1] = (
#            (T[2:,1:-1] - 2*T[1:-1,1:-1] + T[:-2,1:-1]) / dx**2 +
#            (T[1:-1,2:] - 2*T[1:-1,1:-1] + T[1:-1,:-2]) / dy**2
#        )
#
#        # enthalpy update (interior only) with heat source
#        H[1:-1,1:-1] += dt * ((k/rho) * Lap[1:-1,1:-1] + Q_Wm3[1:-1,1:-1] / rho)
#
#        # phase update
#        f_new = (H - cp*Tm) / L_lat
#        mush  = (f_new > 0.0) & (f_new < 1.0)
#        solid = (f_new <= 0.0)
#        liq   = (f_new >= 1.0)
#
#        T[mush]  = Tm
#        f[mush]  = f_new[mush]
#        T[solid] = H[solid]/cp
#        f[solid] = 0.0
#        T[liq]   = (H[liq]-L_lat)/cp
#        f[liq]   = 1.0
#
#        # re-apply BCs and resync H on Dirichlet sides
#        apply_BCs(T, H)
#
#        # save snapshots
#        if next_k < len(save_times) and t >= save_times[next_k] - 0.5*dt:
#            Ts.append(T.copy()); Fs.append(f.copy()); ts.append(t); next_k += 1
#
#        t += dt
#
#    return np.array(Ts), np.array(Fs), np.array(ts), X, Y, {"dt": dt}

In [ ]:
# --------------------------
# 3) Battery heat-source builder (Q + battery mask)
#    IMPORTANT: arrays are in solver indexing [nx,ny] (x first, y second)
# --------------------------
def sample_batteries_hw(
    rng, nx, ny,
    Nb_range=(2, 5),
    size_range=(7, 15),          # square side length in cells
    q_range=(5.0e5, 8.0e5),      # W/m^3 inside batteries
    margin=3,
    allow_overlap=False,
    max_tries=6000
):
    Nb = int(rng.integers(Nb_range[0], Nb_range[1] + 1))
    bats = []
    tries = 0

    def rect(cx, cy, s):
        half = s // 2
        x0 = cx - half
        x1 = x0 + s
        y0 = cy - half
        y1 = y0 + s
        return x0, x1, y0, y1

    def overlaps(r1, r2):
        x0, x1, y0, y1 = r1
        a0, a1, b0, b1 = r2
        return not (x1 <= a0 or a1 <= x0 or y1 <= b0 or b1 <= y0)

    while len(bats) < Nb and tries < max_tries:
        tries += 1
        s = int(rng.integers(size_range[0], size_range[1] + 1))
        if s % 2 == 0:
            s += 1
        half = s // 2

        cx = int(rng.integers(margin + half, nx - margin - half))
        cy = int(rng.integers(margin + half, ny - margin - half))
        q  = float(rng.uniform(q_range[0], q_range[1]))

        r_new = rect(cx, cy, s)

        if not allow_overlap:
            ok = True
            for b in bats:
                r_old = rect(b["cx"], b["cy"], b["s"])
                if overlaps(r_new, r_old):
                    ok = False
                    break
            if not ok:
                continue

        bats.append({"cx": cx, "cy": cy, "s": s, "q": q})

    return bats

def _avg_blur_3x3(A):
    Ap = np.pad(A, ((1,1),(1,1)), mode="edge")
    out = (
        Ap[0:-2, 0:-2] + Ap[0:-2, 1:-1] + Ap[0:-2, 2:] +
        Ap[1:-1, 0:-2] + Ap[1:-1, 1:-1] + Ap[1:-1, 2:] +
        Ap[2:,   0:-2] + Ap[2:,   1:-1] + Ap[2:,   2:]
    ) / 9.0
    return out.astype(np.float32)

def render_battery_Q_and_mask(nx, ny, batteries, base=4.5e5, soft_edges=True, blur_iters=1):
    """
    Returns:
      Q   : [nx,ny] W/m^3
      M   : [nx,ny] 1=battery pixels
    base gives a background level (like your example image top row).
    """
    Q = np.full((nx, ny), float(base), dtype=np.float32)
    M = np.zeros((nx, ny), dtype=np.float32)

    for b in batteries:
        cx, cy, s, q = b["cx"], b["cy"], b["s"], b["q"]
        half = s // 2
        x0 = cx - half
        x1 = x0 + s
        y0 = cy - half
        y1 = y0 + s

        Q[x0:x1, y0:y1] = q
        M[x0:x1, y0:y1] = 1.0

    if soft_edges:
        for _ in range(max(1, int(blur_iters))):
            Q = _avg_blur_3x3(Q)

    return Q.astype(np.float32), M.astype(np.float32)

In [ ]:
# --------------------------
# 4) Plot in your 3-row style
#    (We plot Q.T / T.T / f.T so x is horizontal and y is vertical)
# --------------------------
def plot_case_Q_T_f(Q, Ts, Fs, times, Lx=0.05, Ly=0.05, bat_mask=None, ncols=7):
    Tn, nx, ny = Ts.shape
    extent = [0.0, Lx, 0.0, Ly]

    idxs = np.linspace(0, Tn-1, min(ncols, Tn)).round().astype(int)

    fig, axs = plt.subplots(3, len(idxs), figsize=(3.2*len(idxs), 8.2))

    # Row 1: Q
    for j, ti in enumerate(idxs):
        ax = axs[0, j]
        im = ax.imshow(Q.T, origin="lower", extent=extent)  # transpose for x-horizontal
        ax.set_title("Heat source Q")
        ax.set_xlabel("x [m]")
        ax.set_ylabel("y [m]")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

        if bat_mask is not None:
            ax.contour(bat_mask.T, levels=[0.5], colors="w", linewidths=1.0,
                       origin="lower", extent=extent)

    # Row 2: Temperature
    for j, ti in enumerate(idxs):
        ax = axs[1, j]
        Tmap = Ts[ti].copy()

        # optional: hide battery region in the plot so maps don't "overlap" batteries visually
        if bat_mask is not None:
            Tmap = Tmap.astype(float)
            Tmap[bat_mask > 0.5] = np.nan

        im = ax.imshow(Tmap.T, origin="lower", extent=extent)
        ax.set_title(f"T at t={times[ti]:.1f} s")
        ax.set_xlabel("x [m]")
        ax.set_ylabel("y [m]")
        cb = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cb.set_label("T [K]")

        if bat_mask is not None:
            ax.contour(bat_mask.T, levels=[0.5], colors="c", linewidths=1.0,
                       origin="lower", extent=extent)

    # Row 3: Liquid fraction
    for j, ti in enumerate(idxs):
        ax = axs[2, j]
        fmap = Fs[ti].copy()

        if bat_mask is not None:
            fmap = fmap.astype(float)
            fmap[bat_mask > 0.5] = np.nan  # no f inside batteries in plot

        im = ax.imshow(fmap.T, origin="lower", vmin=0.0, vmax=1.0, extent=extent)
        ax.set_title(f"Liquid fraction at t={times[ti]:.1f} s")
        ax.set_xlabel("x [m]")
        ax.set_ylabel("y [m]")
        cb = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cb.set_label("f [-]")

        if bat_mask is not None:
            ax.contour(bat_mask.T, levels=[0.5], colors="r", linewidths=1.0,
                       origin="lower", extent=extent)

    plt.tight_layout()
    plt.show()

In [ ]:
    # ============================================================
    # 5) RUN ONE CASE (this produces exactly 3 rows like your screenshot)
    # ============================================================
for i in range(10,15):
     N = 64
     Lx = 0.05
     Ly = 0.05
     
     # Make "interesting" BCs (you can change these to match your desired style)
     bc = {
         "left":   {"on": True,  "type": "const", "T": 370.0},
         "right":  {"on": True,  "type": "const", "T": 300.0},  # adiabatic
         "bottom": {"on": True,  "type": "const", "T": 300.0},
         "top":    {"on": True,  "type": "const", "T": 300.0},
     }
     
     # Battery Q map
     rng = np.random.default_rng(i)
     bats = sample_batteries_hw(
         rng, nx=N, ny=N,
         Nb_range=(3, 3),
         size_range=(14,17),
         q_range=(5e5, 8e5),
         margin=3,
         allow_overlap=False
     )
     Q, bat_mask = render_battery_Q_and_mask(
         nx=N, ny=N, batteries=bats,
         base=1e5,         # background level like your shown top row
         soft_edges=False,
         blur_iters=1
     )
     
     # Times to save (7 columns like your screenshot)
     save_times = (10.0, 60.0, 300.0, 600.0, 1200.0, 2000.0, 2500.0)
     
     Ts, Fs, ts, X, Y, info = simulate_pcm_2d_with_source(
         Q_Wm3=Q,
         nx=N, ny=N, Lx=Lx, Ly=Ly,
         rho=800.0, cp=2000.0, k=0.2, L_lat=2e5, Tm=330.0,
         T_init=300.0, Tb=300.0,
         t_end=2500.0, cfl=0.45,
         save_times=save_times,
         bc=bc,
         battery_mask=bat_mask,   # key: no melting inside batteries
     )
     
     print("Saved times:", ts)
     print("dt:", info["dt"])
     print("Batteries:", bats)
     
     plot_case_Q_T_f(Q, Ts, Fs, ts, Lx=Lx, Ly=Ly, bat_mask=bat_mask, ncols=7)

In [ ]:
def plot_fields_with_Q(Ts, Fs, ts, X, Y, Tm, Q, qlabel="Q [W/m^3]",colormap="RdBu_r"):
    """
    Show heat source Q plus temperature and liquid fraction snapshots.
    Rows: 0=Q (same in all columns), 1=T, 2=f
    Columns: different saved times.
    """
    n = len(ts)
    fig, axes = plt.subplots(3, n, figsize=(4.4*n, 10), dpi=120, constrained_layout=True)

    x0, x1 = float(X.min()), float(X.max())
    y0, y1 = float(Y.min()), float(Y.max())

    #Q row 0
    for k in range(n):
        imQ = axes[0, k].imshow(Q.T, origin="lower",
                                extent=[x0, x1, y0, y1],
                                cmap=colormap,
                                aspect="equal")
        axes[0, k].set_title("Heat source Q")
        axes[0, k].set_xlabel("x [m]"); axes[0, k].set_ylabel("y [m]")
        if k == 0:
            fig.colorbar(imQ, ax=axes[0, k], shrink=0.85, label=qlabel)
        else:
            fig.colorbar(imQ, ax=axes[0, k], shrink=0.85)

    # temperature with T=Tm contour row 1
    for k in range(n):
        imT = axes[1, k].imshow(Ts[k].T, origin="lower",
                                extent=[x0, x1, y0, y1],
                                cmap="inferno", aspect="equal")
        axes[1, k].contour(X, Y, Ts[k], levels=[Tm], colors='cyan', linewidths=1.0)
        axes[1, k].set_title(f"T at t={ts[k]:.1f} s")
        axes[1, k].set_xlabel("x [m]"); axes[1, k].set_ylabel("y [m]")
        fig.colorbar(imT, ax=axes[1, k], shrink=0.85, label="T [K]")

    #Liquid fraction f row2
    for k in range(n):
        imF = axes[2, k].imshow(Fs[k].T, origin="lower",
                                extent=[x0, x1, y0, y1],
                                cmap="viridis", vmin=0.0, vmax=1.0, aspect="equal")
        axes[2, k].set_title(f"Liquid fraction at t={ts[k]:.1f} s")
        axes[2, k].set_xlabel("x [m]"); axes[2, k].set_ylabel("y [m]")
        fig.colorbar(imF, ax=axes[2, k], shrink=0.85, label="f [-]")

    plt.show()

In [ ]:
'''# --- knobs you already use ---
T_m      = 330.0
t_end    = 4000.0
T_bound  = 350.0
T_initial= 300.0

N  = 120
nx = ny = N
Lx = Ly = 0.05
x  = np.linspace(0.0, Lx, nx)
y  = np.linspace(0.0, Ly, ny)

rho, cp, k, L_lat = 800.0, 2000.0, 0.2, 2e5
cfl = 0.45
save_times = (10.0, 60.0, 300.0, 600.0, 1200.0, 2000.0, 2500.0)

q_scale = 5e4   # W/m^3
Q_length_scale = 0.18
Q_sigma        = 1.0

# ------------ helper: plot boundary profiles (reuses your _make_profile) ------------
def plot_boundary_profiles(bc, x, y, Lx, Ly, figtitle=None, savepath=None):
    onL, TL = _make_profile(bc.get("left",   {}), y, Ly)
    onR, TR = _make_profile(bc.get("right",  {}), y, Ly)
    onB, TB = _make_profile(bc.get("bottom", {}), x, Lx)
    onT, TT = _make_profile(bc.get("top",    {}), x, Lx)

    fig, ax = plt.subplots(2, 2, figsize=(8,6), constrained_layout=True)
    if figtitle: fig.suptitle(figtitle)

    if onL: ax[0,0].plot(y, TL, lw=2); ax[0,0].set_title("Left (x=0)")
    else:   ax[0,0].set_title("Left OFF (adiabatic)")
    ax[0,0].set_xlabel("y [m]"); ax[0,0].set_ylabel("T [K]")

    if onR: ax[0,1].plot(y, TR, lw=2); ax[0,1].set_title("Right (x=Lx)")
    else:   ax[0,1].set_title("Right OFF (adiabatic)")
    ax[0,1].set_xlabel("y [m]"); ax[0,1].set_ylabel("T [K]")

    if onB: ax[1,0].plot(x, TB, lw=2); ax[1,0].set_title("Bottom (y=0)")
    else:   ax[1,0].set_title("Bottom OFF (adiabatic)")
    ax[1,0].set_xlabel("x [m]"); ax[1,0].set_ylabel("T [K]")

    if onT: ax[1,1].plot(x, TT, lw=2); ax[1,1].set_title("Top (y=Ly)")
    else:   ax[1,1].set_title("Top OFF (adiabatic)")
    ax[1,1].set_xlabel("x [m]"); ax[1,1].set_ylabel("T [K]")

    for a in ax.ravel(): a.grid(alpha=0.25)
    if savepath:
        plt.savefig(savepath, dpi=200, bbox_inches="tight")
    plt.show()

# ------------ boundary-case generator (10 combos) ------------
def make_bc_cases(T_bound, Lx, Ly):
    cases = []

    # 1) All sides hot (const), left as Gaussian
    cases.append({
        "left":   {"on": True, "type": "gauss", "base":T_bound, "amp":20, "mu":0.5, "sigma":0.20},
        "right":  {"on": True, "type": "const", "T": T_bound},
        "bottom": {"on": True, "type": "const", "T": T_bound},
        "top":    {"on": True, "type": "const", "T": T_bound},
    })

    # 2–4) One-side hot each time
    cases.append({"left":  {"on": True, "type": "const", "T": T_bound},
                  "right": {"on": False}, "bottom":{"on": False}, "top":{"on": False}})
    cases.append({"right": {"on": True, "type": "const", "T": T_bound},
                  "left":  {"on": False}, "bottom":{"on": False}, "top":{"on": False}})
    cases.append({"bottom":{"on": True, "type": "const", "T": T_bound},
                  "left":  {"on": False}, "right":{"on": False}, "top":{"on": False}})

    # 5) Left Gaussian, top const, others off
    cases.append({"left":  {"on": True, "type": "gauss", "base":T_bound, "amp":20, "mu":0.5, "sigma":0.20},
                  "top":   {"on": True, "type": "const", "T": T_bound},
                  "right": {"on": False}, "bottom":{"on": False}})

    # 6–8) Vary left Gaussian position/width
    cases.append({"left":  {"on": True, "type": "gauss", "base":T_bound, "amp":20, "mu":0.3, "sigma":0.15},
                  "right": {"on": True, "type": "const", "T": T_bound},
                  "bottom":{"on": False}, "top":{"on": False}})
    cases.append({"left":  {"on": True, "type": "gauss", "base":T_bound, "amp":20, "mu":0.7, "sigma":0.25},
                  "bottom":{"on": True, "type": "const", "T": T_bound},
                  "right": {"on": False}, "top":{"on": False}})
    cases.append({"left":  {"on": True, "type": "gauss", "base":T_bound, "amp":25, "mu":0.5, "sigma":0.10},
                  "top":   {"on": True, "type": "const", "T": T_bound},
                  "right": {"on": False}, "bottom":{"on": False}})

    # 9) Opposite sides hot (right & bottom), left Gaussian off
    cases.append({"left":  {"on": False},
                  "right": {"on": True, "type": "const", "T": T_bound},
                  "bottom":{"on": True, "type": "const", "T": T_bound},
                  "top":   {"on": False}})

    # 10) Checker: left Gaussian + right const + top const
    cases.append({"left":  {"on": True, "type": "gauss", "base":T_bound, "amp":15, "mu":0.6, "sigma":0.18},
                  "right": {"on": True, "type": "const", "T": T_bound},
                  "top":   {"on": True, "type": "const", "T": T_bound},
                  "bottom":{"on": False}})
    return cases

bc_cases = make_bc_cases(T_bound, Lx, Ly)

# ------------ batch loop ------------
for i, bc in enumerate(bc_cases, 1):
    title = f"BC Case {i}"
    # 1) Boundary profiles
    plot_boundary_profiles(bc, x, y, Lx, Ly, figtitle=title,
                           savepath=f"bc_profiles_case{i}.png")

    # 2) Positive heat source (static in time)
    genQ = HeatSource2DRBFPosterior(grid_size=N, length=1.0,
                                    length_scale=Q_length_scale, sigma=Q_sigma,
                                    jitter=1e-6, seed=1234+i)  # seed per case for variety

    Q_dimless = genQ.sample_posterior(
        enforce_zero_mean=False,
        force_sign='positive',     # strictly >=0
        target_mean=1.0,           # dimensionless mean ≈ 1
        keep_boundary_zero=False   # allow interior & edges to carry source
    )
    Q = q_scale * Q_dimless

    # 3) Simulate
    Ts, Fs, ts, X, Y, info = simulate_pcm_2d_with_source(
        Q_Wm3=Q, nx=N, ny=N, Lx=Lx, Ly=Ly,
        rho=rho, cp=cp, k=k, L_lat=L_lat,
        t_end=t_end, cfl=cfl, bc=bc, T_init=T_initial, Tm=T_m,
        save_times=save_times
    )

    # 4) Plot maps (use your function signature; change colormap as you like)
    plot_fields_with_Q(Ts, Fs, ts, X, Y, T_m, Q, qlabel="Q [W/m^3]", colormap="Reds")
    plt.savefig(f"maps_case{i}.png", dpi=200, bbox_inches="tight")
    plt.close('all')'''


In [ ]:
#T_m       = 330.0
#t_end     = 4000.0
#T_bound   = 350.0
#T_initial = 300.0
#
#N  = 120
#nx = ny = N
#Lx = Ly = 0.05
#x  = np.linspace(0.0, Lx, nx)
#y  = np.linspace(0.0, Ly, ny)
#
#rho, cp, k, L_lat = 800.0, 2000.0, 0.2, 2e5
#cfl = 0.45
#save_times = (10.0, 60.0, 300.0, 600.0, 1200.0, 2000.0, 2500.0)
#
## heat-source GP params
#q_scale        = 5e4       # W/m^3
#Q_length_scale = 0.18
#Q_sigma        = 1.0
#
#
#def plot_boundary_profiles(bc, x, y, Lx, Ly, figtitle=None, savepath=None):
#    onL, TL = _make_profile(bc.get("left",   {}), y, Ly)
#    onR, TR = _make_profile(bc.get("right",  {}), y, Ly)
#    onB, TB = _make_profile(bc.get("bottom", {}), x, Lx)
#    onT, TT = _make_profile(bc.get("top",    {}), x, Lx)
#
#    fig, ax = plt.subplots(2, 2, figsize=(8,6), constrained_layout=True)
#    if figtitle: fig.suptitle(figtitle)
#
#    ax[0,0].plot(y, TL, lw=2); ax[0,0].set_title("Left (x=0)")
#    ax[0,1].plot(y, TR, lw=2); ax[0,1].set_title("Right (x=Lx)")
#    ax[1,0].plot(x, TB, lw=2); ax[1,0].set_title("Bottom (y=0)")
#    ax[1,1].plot(x, TT, lw=2); ax[1,1].set_title("Top (y=Ly)")
#
#    for a in ax.ravel():
#        a.set_xlabel(("y [m]" if a in (ax[0,0], ax[0,1]) else "x [m]"))
#        a.set_ylabel("T [K]")
#        a.grid(alpha=0.25)
#
#    if savepath:
#        plt.savefig(savepath, dpi=200, bbox_inches="tight")
#    plt.show()
#
## make 15 BC cases (ALL sides ON)
#def make_bc_cases_all_on(T_bound, n_cases=15):
#    # cyclic parameter grids (repeat as needed)
#    mus    = np.linspace(0.25, 0.75, 5)          # Gaussian centers (relative)
#    sigmas = np.linspace(0.12, 0.28, 3)          # Gaussian widths (relative)
#    amps   = [12.0, 18.0, 24.0]                  # amplitude (K)
#
#    cases = []
#    for i in range(n_cases):
#        # choose parameters cyclically for variety
#        muL,  muR  = mus[i % len(mus)], mus[(i+2) % len(mus)]
#        muB,  muT  = mus[(i+1) % len(mus)], mus[(i+3) % len(mus)]
#        sL,   sR   = sigmas[i % len(sigmas)], sigmas[(i+1) % len(sigmas)]
#        sB,   sT   = sigmas[(i+2) % len(sigmas)], sigmas[(i+0) % len(sigmas)]
#        aL,   aR   = amps[i % len(amps)], amps[(i+1) % len(amps)]
#        aB,   aT   = amps[(i+2) % len(amps)], amps[(i+0) % len(amps)]
#
#        # alternate sides between Gaussian and constant—but ON for all
#        if i % 3 == 0:
#            left   = {"on": True, "type": "gauss", "base": T_bound, "amp": aL, "mu": muL, "sigma": sL}
#            right  = {"on": True, "type": "const", "T": T_bound}
#            bottom = {"on": True, "type": "gauss", "base": T_bound, "amp": aB, "mu": muB, "sigma": sB}
#            top    = {"on": True, "type": "const", "T": T_bound}
#        elif i % 3 == 1:
#            left   = {"on": True, "type": "const", "T": T_bound}
#            right  = {"on": True, "type": "gauss", "base": T_bound, "amp": aR, "mu": muR, "sigma": sR}
#            bottom = {"on": True, "type": "const", "T": T_bound}
#            top    = {"on": True, "type": "gauss", "base": T_bound, "amp": aT, "mu": muT, "sigma": sT}
#        else:
#            left   = {"on": True, "type": "gauss", "base": T_bound, "amp": aL, "mu": muL, "sigma": sL}
#            right  = {"on": True, "type": "gauss", "base": T_bound, "amp": aR, "mu": muR, "sigma": sR}
#            bottom = {"on": True, "type": "const", "T": T_bound}
#            top    = {"on": True, "type": "const", "T": T_bound}
#
#        cases.append({"left": left, "right": right, "bottom": bottom, "top": top})
#    return cases
#
#bc_cases = make_bc_cases_all_on(T_bound, n_cases=15)
#
## ---------- batch loop ----------
#for i, bc in enumerate(bc_cases, 1):
#    title = f"BC Case {i:02d} (all sides ON)"
#    plot_boundary_profiles(bc, x, y, Lx, Ly, figtitle=title,
#                           savepath=f"bc_profiles_case{i:02d}.png")
#
#    # positive heat-source (static in time), new seed per case
#    genQ = HeatSource2DRBFPosterior(
#        grid_size=N, length=1.0,
#        length_scale=Q_length_scale, sigma=Q_sigma,
#        jitter=1e-6, seed=1000+i
#    )
#    Q_dimless = genQ.sample_posterior(
#        enforce_zero_mean=False,
#        force_sign='positive',    # strictly >= 0
#        target_mean=1.0,
#        keep_boundary_zero=False  # source not forced to zero on edges
#    )
#    Q = q_scale * Q_dimless
#
#    Ts, Fs, ts, X, Y, info = simulate_pcm_2d_with_source(
#        Q_Wm3=Q, nx=N, ny=N, Lx=Lx, Ly=Ly,
#        rho=rho, cp=cp, k=k, L_lat=L_lat,
#        t_end=t_end, cfl=cfl, bc=bc, T_init=T_initial, Tm=T_m,
#        save_times=save_times
#    )
#
#    plot_fields_with_Q(Ts, Fs, ts, X, Y, T_m, Q, qlabel="Q [W/m^3]", colormap="Reds")
#    plt.savefig(f"maps_case{i:02d}.png", dpi=200, bbox_inches="tight")
#    plt.close('all')

In [ ]:
'''def simulate_pcm_2d_with_source(
    Q_Wm3, nx=128, ny=128, Lx=0.05, Ly=0.05,
    rho=800.0, cp=2000.0, k=0.2, L_lat=2e5, Tm=330.0,
    T_init=300.0, Tb=300.0,
    t_end=1200.0, cfl=0.45, save_times=(60.0, 300.0, 600.0, 1200.0)
):
    # grid
    x = np.linspace(0.0, Lx, nx); y = np.linspace(0.0, Ly, ny)
    dx = x[1] - x[0]; dy = y[1] - y[0]
    X, Y = np.meshgrid(x, y, indexing="ij")

    # fields
    T = np.full((nx, ny), T_init, dtype=np.float64)
    f = np.zeros_like(T)
    H = cp*T + f*L_lat

    # to impose Dirichlet temperature on all walls once before the loop
    T[0,:]=T[-1,:]=Tb; T[:,0]=T[:,-1]=Tb
    H[0,:]=cp*Tb + f[0,:]*L_lat
    H[-1,:]=cp*Tb + f[-1,:]*L_lat
    H[:,0]=cp*Tb + f[:,0]*L_lat
    H[:,-1]=cp*Tb + f[:,-1]*L_lat

    # explicit stability bound for 2D FTCS:  that 1/4(dx^2+dy^2)/alpha condition taught in cfd
    alpha = k/(rho*cp)
    dt = cfl * min(dx*dx, dy*dy) / (4.0*alpha)

    save_times = np.array(sorted(save_times), dtype=float)
    Ts, Fs, ts, next_k = [], [], [], 0
    t = 0.0

    while t < t_end + 1e-12:
        # to always enforce wall temperature before forming Laplacian as a safety checker
        T[0,:]=T[-1,:]=Tb; T[:,0]=T[:,-1]=Tb

        # five-point Laplacian-interior only formation
        Lap = np.zeros_like(T)
        Lap[1:-1,1:-1] = (
            (T[2:,1:-1] - 2*T[1:-1,1:-1] + T[:-2,1:-1]) / dx**2 +
            (T[1:-1,2:] - 2*T[1:-1,1:-1] + T[1:-1,:-2]) / dy**2
        )

        #enthalpy update: interior only, with heat source
        H[1:-1,1:-1] += dt * ((k/rho) * Lap[1:-1,1:-1] + Q_Wm3[1:-1,1:-1] / rho)

        #returns boolean arrays here in 2d space as masks
        f_new = (H - cp*Tm) / L_lat
        mush  = (f_new > 0.0) & (f_new < 1.0)
        solid = (f_new <= 0.0)
        liq   = (f_new >= 1.0)

        T[mush]  = Tm;
        f[mush]  = f_new[mush]
        T[solid] = H[solid]/cp
        f[solid] = 0.0
        T[liq]   = (H[liq]-L_lat)/cp
        f[liq] = 1.0

        #reimpose Dirichlet temperature on walls and resync H values
        T[0,:]=T[-1,:]=Tb; T[:,0]=T[:,-1]=Tb
        H[0,:]  = cp*Tb + f[0,:]*L_lat
        H[-1,:] = cp*Tb + f[-1,:]*L_lat
        H[:,0]  = cp*Tb + f[:,0]*L_lat
        H[:,-1] = cp*Tb + f[:,-1]*L_lat

        #save snapshots
        if next_k < len(save_times) and t >= save_times[next_k] - 0.5*dt:
            Ts.append(T.copy()); Fs.append(f.copy()); ts.append(t); next_k += 1

        t += dt

    return np.array(Ts), np.array(Fs), np.array(ts), X, Y, {"dt": dt}

def plot_fields_with_Q(Ts, Fs, ts, X, Y, Tm, Q, qlabel="Q [W/m^3]"):
    """
    Show heat source Q plus temperature and liquid fraction snapshots.
    Rows: 0=Q (same in all columns), 1=T, 2=f
    Columns: different saved times.
    """
    n = len(ts)
    fig, axes = plt.subplots(3, n, figsize=(4.4*n, 10), dpi=120, constrained_layout=True)

    x0, x1 = float(X.min()), float(X.max())
    y0, y1 = float(Y.min()), float(Y.max())

    #Q row 0
    for k in range(n):
        imQ = axes[0, k].imshow(Q.T, origin="lower",
                                extent=[x0, x1, y0, y1],
                                cmap="RdBu_r", aspect="equal")
        axes[0, k].set_title("Heat source Q")
        axes[0, k].set_xlabel("x [m]"); axes[0, k].set_ylabel("y [m]")
        if k == 0:
            fig.colorbar(imQ, ax=axes[0, k], shrink=0.85, label=qlabel)
        else:
            fig.colorbar(imQ, ax=axes[0, k], shrink=0.85)

    # temperature with T=Tm contour row 1
    for k in range(n):
        imT = axes[1, k].imshow(Ts[k].T, origin="lower",
                                extent=[x0, x1, y0, y1],
                                cmap="inferno", aspect="equal")
        axes[1, k].contour(X, Y, Ts[k], levels=[Tm], colors='cyan', linewidths=1.0)
        axes[1, k].set_title(f"T at t={ts[k]:.1f} s")
        axes[1, k].set_xlabel("x [m]"); axes[1, k].set_ylabel("y [m]")
        fig.colorbar(imT, ax=axes[1, k], shrink=0.85, label="T [K]")

    #Liquid fraction f row2
    for k in range(n):
        imF = axes[2, k].imshow(Fs[k].T, origin="lower",
                                extent=[x0, x1, y0, y1],
                                cmap="viridis", vmin=0.0, vmax=1.0, aspect="equal")
        axes[2, k].set_title(f"Liquid fraction at t={ts[k]:.1f} s")
        axes[2, k].set_xlabel("x [m]"); axes[2, k].set_ylabel("y [m]")
        fig.colorbar(imF, ax=axes[2, k], shrink=0.85, label="f [-]")

    plt.show()'''


In [ ]:
'''if __name__ == "__main__":
    N = 40
    genQ = HeatSource2DRBFPosterior(grid_size=N, length=1.0, length_scale=0.18, sigma=1.0, jitter=1e-6, seed=None)
    Q_dimless = genQ.sample_posterior(enforce_zero_mean=False, force_sign="positive",target_mean=5e4,keep_boundary_zero=True)

    #scaling to physical W/m^3 magnitude for source strength
    #tune the parameters as needed here
    q_scale = 5e4    # W/m^3
    Q = q_scale * Q_dimless
    T_m=330
    t_end=4000.0
    T_bound=350.0
    T_initial=300.0

    Ts, Fs, ts, X, Y, info = simulate_pcm_2d_with_source(
        Q_Wm3=Q, nx=N, ny=N, Lx=0.05, Ly=0.05,
        rho=800.0, cp=2000.0, k=0.2, L_lat=3e5, Tm=T_m,
        T_init=T_initial,Tb=T_bound, t_end=t_end, cfl=0.45, save_times=(10.0,60.0, 300.0, 600.0, 1200.0, 2000.0,3000.0,3900.0)
    )
    print(f"dt = {info['dt']:.4e} s")
    plot_fields_with_Q(Ts, Fs, ts, X, Y, Tm=T_m,Q=Q, qlabel="Q [W/m³]")'''

### DATASET GENERATOR


In [ ]:
N        = 40          # grid points in x,y (Q, T, f are NxN)
Lx, Ly   = 0.05, 0.05
rho, cp, k, L_lat = 800.0, 2000.0, 0.2, 2e5
T_m      = 330.0
T_init   = 300.0
T_bound  = 330.0
t_end    = 4000.0
save_times = (100.0, 200.0, 300.0, 400.0, 500.0, 600.0, 700.0, 800.0, 900.0, 1000.0, 1100.0, 1200.0, 1300.0, 1400.0, 1500.0, 1600.0, 1700.0, 1800.0, 1900.0, 2000.0, 2100.0, 2500.0, 3000.0, 3500.0)
cfl      = 0.45

# Heat-source GP
q_scale        = 1e5     # W/m^3
Q_length_scale = 0.18
Q_sigma        = 1.0

# Boundary Conditions
mu_max=0.8
sigma_max=0.7
amp_max=80

# Dataset sizes
NUM_CASES      = 200.0      # total function-realizations / cases
TRAIN_FRAC     = 0.8
VAL_FRAC       = 0.1      # test is the remainder

# DeepONet sampling
sensor_mode    = "full"   # "full" (flatten full NxN) or "downsample"
S_down         = 40       # used if sensor_mode="downsample" (S_down x S_down grid)
n_time_bc      = len(save_times)
S_down_BC      = N

points_per_case_per_time = N**2  # # of (x,y) points sampled per time snapshot

# Boundary configuration control
only_lr_vary   = True     # True: top & bottom constant, left/right varied; False: allow all configurable
All_side_const_temp_boundary = False    # if all sides constant boundary is wanted at T_bound for dataset testing

# MODEL PARAMETERS

BATCH_POINTS = 32768//2
EPOCHS = 60
LR = 1e-3
WEIGHT_DECAY = 1e-4# first case 0.0
VAL_SAMPLES = 4
VAL_BATCH_POINTS = 32768//2
STEPS_PER_EPOCH = 64

#each path weight
alpha_add=1.0
alpha_prod=0.2

Q_MODE_PROBS = {"gp": 0.33, "battery": 0.33, "hybrid": 0.34}

BAT_NB_RANGE   = (2, 5)
BAT_SIZE_RANGE = (5, 16)         # in cells (keeps your current style)
BAT_Q_RANGE    = (1.0e6, 1.2e6)
BAT_MARGIN     = 3
BAT_SOFT_EDGES = True
BAT_BLUR_ITERS = 1
BAT_BASE       = 4.5

HYB_W_GP_RANGE  = (0.4, 0.7)
HYB_W_BAT_RANGE = (0.3, 0.6)


In [ ]:
# Output directory
RUN_DIR = Path(f"dataset_run_{time.strftime('%Y%m%d-%H%M%S')}")
RUN_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def make_bc_case(i, T_bound=350.0, only_lr=True, all_const=False):
    """
    i: case index for deterministic variety
    only_lr=True -> top/bottom constant; left/right varied (gaussian/const)
    """
    rng = np.random.default_rng(10_000 + i)

    # const sides (top, bottom)
    bottom = {"on": True, "type": "const", "T": T_bound}
    top    = {"on": True, "type": "const", "T": T_bound}

    # varied sides: either const or gaussian around T_bound
    def rand_gauss():
        mu    = rng.uniform(0.25, mu_max)
        sigma = rng.uniform(0.12, sigma_max)
        amp   = rng.uniform(12.0, amp_max)
        return {"on": True, "type": "gauss", "base": T_bound, "amp": float(amp),
                "mu": float(mu), "sigma": float(sigma)}

    def rand_const():
        # small jitter around T_bound if you like; here keep exactly T_bound
        return {"on": True, "type": "const", "T": T_bound}

    left  = rand_gauss() if (i % 2 == 0) else rand_const()
    right = rand_gauss() if (i % 3 == 0) else rand_const()

    if not only_lr:
        # Optionally make top/bottom also vary (still ON)
        if (i % 5) == 0:
            top = rand_gauss()
        if (i % 7) == 0:
            bottom = rand_gauss()

    if all_const:
        bottom = {"on": True, "type": "const", "T": T_bound}
        top    = {"on": True, "type": "const", "T": T_bound}
        right = {"on": True, "type": "const", "T": T_bound}
        left = {"on": True, "type": "const", "T": T_bound}

    return {"left": left, "right": right, "bottom": bottom, "top": top}

# --- helper: branch sensorizing ---
def sensorize_field(arr2d, mode="full", S=40):
    """
    Converts a 2D field (NxN) to a branch vector.
    mode="full": flatten (N*N,)
    mode="downsample": sample on an SxS regular grid
    """
    if mode == "full":
        return arr2d.reshape(-1).astype(np.float32)
    elif mode == "downsample":
        N = arr2d.shape[0]
        xs = np.linspace(0, N-1, S).round().astype(int)
        ys = np.linspace(0, N-1, S).round().astype(int)
        sub = arr2d[np.ix_(xs, ys)]
        return sub.reshape(-1).astype(np.float32)
    else:
        raise ValueError("sensor_mode must be 'full' or 'downsample'")

# --- helper: sample trunk points for a case ---
#def sample_trunk_points(N, Lx, Ly, times, per_time, rng):
#    """
#    Returns: coords [M,3] (x,y,t), idxs [M,2] (i,j indices on grid), time_ids [M]
#    where M = per_time * len(times)
#    """
#    xs = np.linspace(0.0, Lx, N)
#    ys = np.linspace(0.0, Ly, N)
#
#    coords_list, ij_list, tid_list = [], [], []
#    for ti, t in enumerate(times):
#        i_idx = rng.integers(0, N, size=per_time)
#        j_idx = rng.integers(0, N, size=per_time)
#        x_samp = xs[i_idx]
#        y_samp = ys[j_idx]
#        t_samp = np.full_like(x_samp, float(t), dtype=np.float64)
#
#        coords_list.append(np.stack([x_samp, y_samp, t_samp], axis=1))
#        ij_list.append(np.stack([i_idx, j_idx], axis=1))
#        tid_list.append(np.full(per_time, ti, dtype=np.int32))
#
#    coords = np.concatenate(coords_list, axis=0)             # [M,3]
#    ij     = np.concatenate(ij_list, axis=0).astype(np.int32)# [M,2]
#    tids   = np.concatenate(tid_list, axis=0).astype(np.int32)
#    return coords, ij, tids
#

def sample_trunk_points_split(N, Lx, Ly, times, per_time, rng):
    """
    Returns:
      xy     [M,2]  -> (x,y)
      t      [M,1]  -> (t,)
      ij     [M,2]  -> (row=j, col=i) for array indexing [j,i]
      tids   [M]    -> time index into `times`
    where M = per_time * len(times)
    """
    if rng is None:
        rng = np.random.default_rng()

    times = np.asarray(times, dtype=float)
    M = per_time * len(times)

    # integer grid indices
    js = rng.integers(0, N, size=M)    # row (y)
    is_ = rng.integers(0, N, size=M)   # col (x)

    # physical coordinates
    dx = Lx / (N - 1 if N > 1 else 1)
    dy = Ly / (N - 1 if N > 1 else 1)
    xs = is_ * dx
    ys = js  * dy

    # times per block
    tids = np.repeat(np.arange(len(times), dtype=np.int32), per_time)
    t    = times[tids].reshape(-1, 1).astype(np.float32)

    xy = np.column_stack([xs, ys]).astype(np.float32)
    ij = np.column_stack([js, is_]).astype(np.int32)
    return xy, t, ij, tids


In [ ]:
import os, json
from pathlib import Path

def plot_boundary_profiles(bc, x, y, Lx, Ly, figtitle=None, savepath=None):
    """
    Line plots of boundary temperature profiles:
      Top (y=Ly)    : vs x
      Bottom (y=0)  : vs x
      Left (x=0)    : vs y
      Right (x=Lx)  : vs y
    Uses your gauss/const BC format. If a side is 'off', it shows NaNs.
    """
    # reuse your profile-maker: returns lambda s in [0,1] or None
    def _pf(side_name):
        side_cfg = bc.get(side_name, {"on": False})
        if not side_cfg.get("on", True):
            return None
        typ = side_cfg.get("type", "const")
        if typ == "const":
            T = float(side_cfg["T"])
            return lambda s: T
        elif typ == "gauss":
            base = float(side_cfg["base"]); amp=float(side_cfg["amp"])
            mu = float(side_cfg["mu"]); sigma=float(side_cfg["sigma"])
            sig2 = 2.0 * (sigma**2)
            return lambda s: base + amp * np.exp(-((s - mu)**2)/sig2)
        else:
            return None

    pf_left  = _pf("left")   # s along y ∈ [0,1]
    pf_right = _pf("right")  # s along y ∈ [0,1]
    pf_bot   = _pf("bottom") # s along x ∈ [0,1]
    pf_top   = _pf("top")    # s along x ∈ [0,1]

    # evaluate
    y_s = (y - y.min()) / max(Ly, 1e-9)
    x_s = (x - x.min()) / max(Lx, 1e-9)

    TL = np.array([pf_left(si)  if pf_left  else np.nan for si in y_s], dtype=float)
    TR = np.array([pf_right(si) if pf_right else np.nan for si in y_s], dtype=float)
    TB = np.array([pf_bot(si)   if pf_bot   else np.nan for si in x_s], dtype=float)
    TT = np.array([pf_top(si)   if pf_top   else np.nan for si in x_s], dtype=float)

    fig, ax = plt.subplots(2, 2, figsize=(8,6), constrained_layout=True)
    if figtitle: fig.suptitle(figtitle)

    ax[0,0].plot(y, TL, lw=2); ax[0,0].set_title("Left (x=0)")
    ax[0,1].plot(y, TR, lw=2); ax[0,1].set_title("Right (x=Lx)")
    ax[1,0].plot(x, TB, lw=2); ax[1,0].set_title("Bottom (y=0)")
    ax[1,1].plot(x, TT, lw=2); ax[1,1].set_title("Top (y=Ly)")

    for i, (t, v) in enumerate(zip(y, TL)):
        if i % 5 == 0 and not np.isnan(v):
            ax[0,0].text(t, v, f"{v:.1f}", fontsize=8, ha='center', va='bottom')
    for i, (t, v) in enumerate(zip(y, TR)):
        if i % 5 == 0 and not np.isnan(v):
            ax[0,1].text(t, v, f"{v:.1f}", fontsize=8, ha='center', va='bottom')

    for a, lbl in zip(ax.ravel(), ["y [m]","y [m]","x [m]","x [m]"]):
        a.set_xlabel(lbl); a.set_ylabel("T [K]"); a.grid(alpha=0.3)

    #plt.plot()
    if savepath: fig.savefig(savepath, dpi=150, bbox_inches="tight")
    plt.close(fig)


def _bc_side_summary(side_name, side_cfg):
    """Compact, human-friendly summary for one BC side."""
    if not side_cfg.get("on", True):
        return {"side": side_name, "on": False, "type": "off"}
    t = side_cfg.get("type", "const")
    if t == "const":
        return {"side": side_name, "on": True, "type": "const", "T": float(side_cfg["T"])}
    elif t == "gauss":
        return {
            "side": side_name, "on": True, "type": "gauss",
            "base": float(side_cfg["base"]),
            "amp": float(side_cfg["amp"]),
            "mu": float(side_cfg["mu"]),
            "sigma": float(side_cfg["sigma"]),
        }
    else:
        # keep generic dump
        d = {"side": side_name, "on": True, "type": t}
        d.update({k: (float(v) if isinstance(v, (int,float)) else v)
                  for k, v in side_cfg.items() if k not in ("on","type")})
        return d

def plot_case_time_stats(TT, FF, times, out_path):
    """
    Plot min/mean/max/var vs time for both T and f (two rows).
    TT, FF: [Nt, N, N]
    """
    Nt = TT.shape[0]
    T_flat = TT.reshape(Nt, -1)
    F_flat = FF.reshape(Nt, -1)

    T_min = T_flat.min(axis=1);  T_mean = T_flat.mean(axis=1)
    T_max = T_flat.max(axis=1);  T_var  = T_flat.var(axis=1)

    f_min = F_flat.min(axis=1);  f_mean = F_flat.mean(axis=1)
    f_max = F_flat.max(axis=1);  f_var  = F_flat.var(axis=1)

    fig, ax = plt.subplots(2, 1, figsize=(8, 6), constrained_layout=True)

    # Temperature
    ax[0].plot(times, T_min,  marker="o", label="min")
    ax[0].plot(times, T_mean, marker="o", label="mean")
    ax[0].plot(times, T_max,  marker="o", label="max")
    for vals in [T_min, T_mean, T_max]:
      for t, v in zip(times, vals):
        ax[0].text(t, v, f"{v:.2f}", fontsize=8, ha='center', va='bottom')

    ax[0].set_title("Temperature stats vs time")
    ax[0].set_xlabel("time (s)"); ax[0].set_ylabel("T [K] / var")
    ax[0].set_ylim(290,900)
    ax[0].grid(True, ls="--", alpha=0.5); ax[0].legend()

    # Liquid fraction
    ax[1].plot(times, f_min,  marker="o", label="min")
    ax[1].plot(times, f_mean, marker="o", label="mean")
    ax[1].plot(times, f_max,  marker="o", label="max")
    #ax[1].plot(times, f_var,  marker="o", label="var")
    for vals in [f_min, f_mean, f_max]:
      for t, v in zip(times, vals):
        ax[1].text(t, v, f"{v:.2f}", fontsize=8, ha='center', va='bottom')
    ax[1].set_title("Liquid fraction stats vs time")
    ax[1].set_xlabel("time (s)"); ax[1].set_ylabel("f [-] / var")
    ax[1].grid(True, ls="--", alpha=0.5); ax[1].legend()

    #plt.plot()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)

    # return stats if you want to persist them per case
    return {
        "T_min": T_min.tolist(), "T_mean": T_mean.tolist(),
        "T_max": T_max.tolist(), "T_var": T_var.tolist(),
        "f_min": f_min.tolist(), "f_mean": f_mean.tolist(),
        "f_max": f_max.tolist(), "f_var": f_var.tolist(),
    }

def plot_case_variance_stats(TT, FF, times, out_path):
    """
    Plot variance vs time for both Temperature and Liquid Fraction.
    """
    Nt = TT.shape[0]
    T_var = TT.reshape(Nt, -1).var(axis=1)
    f_var = FF.reshape(Nt, -1).var(axis=1)

    fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
    ax.plot(times, T_var, marker="o", label="T variance")
    ax.plot(times, f_var, marker="o", label="f variance")

    # data labels
    for vals in [T_var, f_var]:
        for t, v in zip(times, vals):
            ax.text(t, v, f"{v:.2e}", fontsize=8, ha='center', va='bottom')

    ax.set_title("Variance vs time")
    ax.set_xlabel("time (s)")
    ax.set_ylabel("Variance")
    ax.grid(True, ls="--", alpha=0.5)
    ax.legend()
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


def _timewise_stats(TT, FF, save_times):
    """
    Per-snapshot stats. Returns a list of dicts with time, T stats, f stats, and melt fractions.
    - melt_frac = fraction of cells with f>0 (any melt)
    - liquid_frac = fraction of cells with f>0.99 (near fully liquid)
    """
    out = []
    Nt = TT.shape[0]
    for k in range(Nt):
        T = TT[k]; f = FF[k]
        ncell = float(T.size)
        melt_frac  = float((f > 0.0).sum()) / ncell
        liquid_frac= float((f > 0.99).sum()) / ncell
        out.append({
            "time": float(save_times[k]),
            "T_min": float(T.min()), "T_mean": float(T.mean()), "T_max": float(T.max()),
            "f_min": float(f.min()), "f_mean": float(f.mean()), "f_max": float(f.max()),
            "melt_frac": melt_frac, "liquid_frac": liquid_frac,
        })
    return out

def write_case_stats(RUN_DIR, case_id, Q, TT, FF, save_times, bc):
    """
    Compute & append concise stats for one case to:
      - RUN_DIR/case_stats.csv    (one row per case, simple columns)
      - RUN_DIR/case_stats.jsonl  (full rich structure per case)

    Returns the full stats dict for in-memory use.
    """
    RUN_DIR = Path(RUN_DIR)
    RUN_DIR.mkdir(parents=True, exist_ok=True)

    # --- Q (heat-source) stats ---
    stats_Q = {
        "Q_min": float(Q.min()),
        "Q_mean": float(Q.mean()),
        "Q_max": float(Q.max()),
        "Q_std": float(Q.std()),
    }

    # --- BC summary (left/right/bottom/top) ---
    bc_summary = [
        _bc_side_summary("left",   bc.get("left",   {"on": False})),
        _bc_side_summary("right",  bc.get("right",  {"on": False})),
        _bc_side_summary("bottom", bc.get("bottom", {"on": False})),
        _bc_side_summary("top",    bc.get("top",    {"on": False})),
    ]

    # --- Timewise T/f stats ---
    per_time = _timewise_stats(TT, FF, save_times)

    # --- assemble full record ---
    record = {
        "case_id": int(case_id),
        "N": int(TT.shape[-1]),
        "Nt": int(TT.shape[0]),
        "times": [float(t) for t in save_times],
        "Q_stats": stats_Q,
        "BC": bc_summary,
        "per_time": per_time,
        # convenient “headline” fields for CSV:
        "T_min_first": per_time[0]["T_min"],
        "T_mean_first": per_time[0]["T_mean"],
        "T_max_last": per_time[-1]["T_max"],
        "f_mean_last": per_time[-1]["f_mean"],
        "melt_frac_last": per_time[-1]["melt_frac"],
        "liquid_frac_last": per_time[-1]["liquid_frac"],
    }

    # --- append JSONL ---
    jsonl_path = RUN_DIR / "case_stats.jsonl"
    with open(jsonl_path, "a", encoding="utf-8") as jf:
        jf.write(json.dumps(record) + "\n")

    # --- append CSV (create header if missing) ---
    csv_path = RUN_DIR / "case_stats.csv"
    csv_cols = [
        "case_id","N","Nt",
        "Q_min","Q_mean","Q_max","Q_std",
        "T_min_first","T_mean_first","T_max_last","f_mean_last","melt_frac_last","liquid_frac_last",
    ]
    make_header = not csv_path.exists()
    with open(csv_path, "a", encoding="utf-8") as cf:
        if make_header:
            cf.write(",".join(csv_cols) + "\n")
        row = [
            str(record["case_id"]),
            str(record["N"]), str(record["Nt"]),
            f'{stats_Q["Q_min"]:.6g}', f'{stats_Q["Q_mean"]:.6g}', f'{stats_Q["Q_max"]:.6g}', f'{stats_Q["Q_std"]:.6g}',
            f'{record["T_min_first"]:.6g}', f'{record["T_mean_first"]:.6g}',
            f'{record["T_max_last"]:.6g}',  f'{record["f_mean_last"]:.6g}',
            f'{record["melt_frac_last"]:.6g}', f'{record["liquid_frac_last"]:.6g}',
        ]
        cf.write(",".join(row) + "\n")

    return record

In [ ]:
def plot_fields_with_Q(Ts, Fs, ts, X, Y, Tm, Q, qlabel="Q [W/m^3]", colormap="RdBu_r", savepath=None):
    """
    Show heat source Q plus temperature and liquid fraction snapshots.
    Rows: 0=Q (same in all columns), 1=T, 2=f
    Columns: different saved times.
    If savepath is given, saves the figure instead of showing it.
    """
    n = len(ts)
    fig, axes = plt.subplots(3, n, figsize=(4.4*n, 10), dpi=120, constrained_layout=True)

    x0, x1 = float(X.min()), float(X.max())
    y0, y1 = float(Y.min()), float(Y.max())

    # Q row 0
    for k in range(n):
        imQ = axes[0, k].imshow(Q.T, origin="lower",
                                extent=[x0, x1, y0, y1],
                                cmap=colormap, aspect="equal")
        axes[0, k].set_title("Heat source Q")
        axes[0, k].set_xlabel("x [m]"); axes[0, k].set_ylabel("y [m]")
        if k == 0:
            fig.colorbar(imQ, ax=axes[0, k], shrink=0.85, label=qlabel)
        else:
            fig.colorbar(imQ, ax=axes[0, k], shrink=0.85)

    # Temperature with T=Tm contour row 1
    for k in range(n):
        imT = axes[1, k].imshow(Ts[k].T, origin="lower",
                                extent=[x0, x1, y0, y1],
                                cmap="inferno", aspect="equal")
        axes[1, k].contour(X, Y, Ts[k], levels=[Tm], colors='cyan', linewidths=1.0)
        axes[1, k].set_title(f"T at t={ts[k]:.1f} s")
        axes[1, k].set_xlabel("x [m]"); axes[1, k].set_ylabel("y [m]")
        fig.colorbar(imT, ax=axes[1, k], shrink=0.85, label="T [K]")

    # Liquid fraction f row 2
    for k in range(n):
        imF = axes[2, k].imshow(Fs[k].T, origin="lower",
                                extent=[x0, x1, y0, y1],
                                cmap="viridis", vmin=0.0, vmax=1.0, aspect="equal")
        axes[2, k].set_title(f"Liquid fraction at t={ts[k]:.1f} s")
        axes[2, k].set_xlabel("x [m]"); axes[2, k].set_ylabel("y [m]")
        fig.colorbar(imF, ax=axes[2, k], shrink=0.85, label="f [-]")

    if savepath:
        fig.savefig(savepath, dpi=150, bbox_inches="tight")
        plt.close(fig)
    else:
        plt.show()

In [ ]:
def _downsample_1d(arr: np.ndarray, n: int) -> np.ndarray:
    if n <= 0:
        return np.empty((0,), dtype=arr.dtype)
    if arr.size == 0:
        return np.zeros((n,), dtype=np.float32)
    idx = np.linspace(0, len(arr) - 1, n).round().astype(int)
    return arr[idx].astype(np.float32)

def make_branch_BC_from_Tsnaps(TT, n_pts_per_side=S_down_BC, n_time_bc=n_time_bc) -> np.ndarray:
    """
    boundary feature = concat over selected times of [top, right, bottom, left] (downsampled)
    S_BC = 4 * n_time_bc * n_pts_per_side
    n_pts_per_side: how many points to sample per boundary side.

    n_time_bc: how many time snapshots to encode. if 3 then taken in regular 3rd interval

    If TT has 7 snapshots, n_time_bc=3, n_pts_per_side=16:
    len(t_idx)=3 , feature length = 4 * 3 * 16 = 192 (float32).
    That is your per-case branch_BC vector.

    """
    if isinstance(TT, np.ndarray) and TT.ndim == 3:
        snaps = [TT[i] for i in range(TT.shape[0])]
    else:
        snaps = list(TT)

    Nt = len(snaps)
    if Nt == 0:
        return np.zeros((4 * n_time_bc * n_pts_per_side,), dtype=np.float32)

    if n_time_bc >= Nt:
        t_idx = np.arange(Nt)
    else:
        t_idx = np.linspace(0, Nt - 1, n_time_bc).round().astype(int)

    pieces = []
    for k in t_idx:
        T = snaps[k]                  # [N, N]
        N = T.shape[0]
        top    = _downsample_1d(T[0,      :], n_pts_per_side)
        right  = _downsample_1d(T[:,  N-1], n_pts_per_side)
        bottom = _downsample_1d(T[N-1,   :], n_pts_per_side)
        left   = _downsample_1d(T[:,      0], n_pts_per_side)
        pieces.extend([*top, *right, *bottom, *left])
    return np.asarray(pieces, dtype=np.float32)

def _ensure_full_Q(Q: np.ndarray, N: int) -> np.ndarray:
    """If Q isn't (N,N), upsample with separable 1D np.interp to avoid broadcast errors in solver."""
    Q = np.asarray(Q, dtype=np.float32)
    if Q.shape == (N, N):
        return Q
    qN_y, qN_x = Q.shape
    xq = np.linspace(0.0, 1.0, qN_x, dtype=np.float32)
    yq = np.linspace(0.0, 1.0, qN_y, dtype=np.float32)
    x  = np.linspace(0.0, 1.0, N,    dtype=np.float32)
    y  = np.linspace(0.0, 1.0, N,    dtype=np.float32)
    # interp along x for each row
    Qx = np.stack([np.interp(x, xq, row) for row in Q], axis=0)           # [qN_y, N]
    # interp along y for each column
    Q_full = np.stack([np.interp(y, yq, Qx[:, j]) for j in range(N)], axis=1)  # [N, N]
    return Q_full.astype(np.float32)

def make_branch_Q_times(Q, Nt, sensor_mode="full", S_down=40):
    """
    Return branch_Q concatenated across Nt snapshots.
    - If Q is a single 2D array: repeats the same vector Nt times.
    - If Q is a list/tuple of 2D arrays (length Nt): encodes each and concatenates.
    Shape: [S_Q * Nt]
    """
    def enc(oneQ):
        if sensor_mode == "full":
            return sensorize_field(oneQ, mode="full")
        else:
            return sensorize_field(oneQ, mode="downsample", S=S_down)

    if isinstance(Q, (list, tuple)):
        assert len(Q) == Nt, "If Q is time-varying you must pass Nt maps."
        vecs = [enc(Qk) for Qk in Q]
    else:
        v = enc(Q)
        vecs = [v] * Nt
    return np.concatenate(vecs, axis=0).astype(np.float32)


In [ ]:
import numpy as np
import json
from pathlib import Path

def _pad_to_multiple(H, W, mult=8):
    """Pad bottom/right so H and W become divisible by mult (for 3 downsamples => mult=8)."""
    pad_h = (mult - (H % mult)) % mult
    pad_w = (mult - (W % mult)) % mult
    return pad_h, pad_w

def _apply_pad(img2d, pad_h, pad_w, mode="constant", constant_values=0.0):
    """
    img2d: [H,W] or [C,H,W]
    Pads bottom and right.
    """
    if img2d.ndim == 2:
        return np.pad(img2d, ((0, pad_h), (0, pad_w)), mode=mode, constant_values=constant_values)
    elif img2d.ndim == 3:
        return np.pad(img2d, ((0, 0), (0, pad_h), (0, pad_w)), mode=mode, constant_values=constant_values)
    else:
        raise ValueError(f"img2d must be 2D or 3D, got shape {img2d.shape}")

def make_bc_maps(bc, X, Y, Lx, Ly):
    """
    Correct for indexing='ij' convention:
      arrays are [nx, ny] with axis-0=x, axis-1=y

    Returns 8 maps (each [H,W]=[nx,ny]):
      left_val,left_mask,right_val,right_mask,bottom_val,bottom_mask,top_val,top_mask
    """
    H, W = X.shape  # H=nx, W=ny

    # 1D axes on the domain
    x_line = X[:, 0]   # length H (x along i)
    y_line = Y[0, :]   # length W (y along j)

    # Prepare empty maps
    left_val   = np.zeros((H, W), dtype=np.float32)
    right_val  = np.zeros((H, W), dtype=np.float32)
    bottom_val = np.zeros((H, W), dtype=np.float32)
    top_val    = np.zeros((H, W), dtype=np.float32)

    left_mask   = np.zeros((H, W), dtype=np.float32)
    right_mask  = np.zeros((H, W), dtype=np.float32)
    bottom_mask = np.zeros((H, W), dtype=np.float32)
    top_mask    = np.zeros((H, W), dtype=np.float32)

    # LEFT boundary: x=0  -> i=0 row, varies along y (j)
    on, prof = _make_profile(bc["left"], axis_array=y_line, axis_len=Ly)
    if on:
        left_val[0, :]  = prof.astype(np.float32)
        left_mask[0, :] = 1.0

    # RIGHT boundary: x=Lx -> i=-1 row, varies along y (j)
    on, prof = _make_profile(bc["right"], axis_array=y_line, axis_len=Ly)
    if on:
        right_val[-1, :]  = prof.astype(np.float32)
        right_mask[-1, :] = 1.0

    # BOTTOM boundary: y=0 -> j=0 col, varies along x (i)
    on, prof = _make_profile(bc["bottom"], axis_array=x_line, axis_len=Lx)
    if on:
        bottom_val[:, 0]  = prof.astype(np.float32)
        bottom_mask[:, 0] = 1.0

    # TOP boundary: y=Ly -> j=-1 col, varies along x (i)
    on, prof = _make_profile(bc["top"], axis_array=x_line, axis_len=Lx)
    if on:
        top_val[:, -1]  = prof.astype(np.float32)
        top_mask[:, -1] = 1.0

    return [
        left_val, left_mask,
        right_val, right_mask,
        bottom_val, bottom_mask,
        top_val, top_mask
    ]


In [ ]:
# ============================================================
# DATA GENERATOR (generalized Q modes: gp / battery / hybrid)
# ============================================================

def build_udeeponet_dataset():
    """
    U-DeepONet dataset (grid-to-grid):
      X_static: [C, Cin, Hpad, Wpad]
      times:    [T]
      Y_T:      [C, T, Hpad, Wpad]
      Y_f:      [C, T, Hpad, Wpad]
    """

    # ----------------- pull globals (keep your style) -----------------
    N        = int(globals()["N"])
    Lx       = float(globals()["Lx"])
    Ly       = float(globals()["Ly"])
    rho      = float(globals()["rho"])
    cp       = float(globals()["cp"])
    k        = float(globals()["k"])
    L_lat    = float(globals()["L_lat"])
    T_m      = float(globals().get("T_m", globals().get("Tm")))
    T_init   = float(globals()["T_init"])
    T_bound  = float(globals()["T_bound"])
    t_end    = float(globals()["t_end"])
    save_times = tuple(globals()["save_times"])
    cfl      = float(globals()["cfl"])

    # GP / Q field params (already in your notebook)
    q_scale        = float(globals()["q_scale"])
    Q_length_scale = float(globals()["Q_length_scale"])
    Q_sigma        = float(globals()["Q_sigma"])

    # dataset sizes
    NUM_CASES      = int(globals()["NUM_CASES"])
    TRAIN_FRAC     = float(globals()["TRAIN_FRAC"])
    VAL_FRAC       = float(globals()["VAL_FRAC"])

    # boundary variety
    only_lr_vary   = bool(globals()["only_lr_vary"])
    All_side_const_temp_boundary = bool(globals()["All_side_const_temp_boundary"])

    ## output dir
    #RUN_DIR        = Path(globals()["RUN_DIR"])
    #RUN_DIR.mkdir(parents=True, exist_ok=True)

    # ----------------- NEW: Q mode mixing knobs (optional globals) -----------------
    # If you don't define these globals, defaults below will be used.
    # Probabilities must sum to 1.0
    Q_MODE_PROBS = globals().get("Q_MODE_PROBS", {"gp": 0.45, "battery": 0.35, "hybrid": 0.20})
    p_gp     = float(Q_MODE_PROBS.get("gp", 0.45))
    p_bat    = float(Q_MODE_PROBS.get("battery", 0.35))
    p_hyb    = float(Q_MODE_PROBS.get("hybrid", 0.20))
    s = p_gp + p_bat + p_hyb
    if abs(s - 1.0) > 1e-6:
        # normalize quietly (so you don't crash if you tweak numbers)
        p_gp, p_bat, p_hyb = p_gp/s, p_bat/s, p_hyb/s

    # battery params (optional globals)
    BAT_NB_RANGE   = globals().get("BAT_NB_RANGE", (2, 5))
    BAT_SIZE_RANGE = globals().get("BAT_SIZE_RANGE", (5, 16))          # in cells (uses your existing sampler)
    BAT_Q_RANGE    = globals().get("BAT_Q_RANGE", (1.0e6, 1.2e6))
    BAT_MARGIN     = int(globals().get("BAT_MARGIN", 3))
    BAT_SOFT_EDGES = bool(globals().get("BAT_SOFT_EDGES", True))
    BAT_BLUR_ITERS = int(globals().get("BAT_BLUR_ITERS", 1))
    BAT_BASE       = float(globals().get("BAT_BASE", 4.5))             # your background

    # hybrid mixing weights (optional globals)
    HYB_W_GP_RANGE  = globals().get("HYB_W_GP_RANGE",  (0.4, 0.7))      # weight for GP part
    HYB_W_BAT_RANGE = globals().get("HYB_W_BAT_RANGE", (0.3, 0.6))      # weight for battery part

    # ----------------- header -----------------
    times = np.array(sorted(save_times), dtype=np.float32)
    T = len(times)
    print("────────────────────────────────────────────────────────")
    print("Building U-DeepONet dataset (grid-to-grid)")
    print(f"N={N} | times={times.tolist()} | NUM_CASES={NUM_CASES}")
    print(f"Q modes probs: gp={p_gp:.2f}, battery={p_bat:.2f}, hybrid={p_hyb:.2f}")
    print(f"Output dir: {RUN_DIR.resolve()}")
    print("────────────────────────────────────────────────────────")

    # ----------------- precompute coordinate maps -----------------
    x = np.linspace(0.0, Lx, N, dtype=np.float32)
    y = np.linspace(0.0, Ly, N, dtype=np.float32)
    X, Y = np.meshgrid(x, y, indexing="ij")  # [N,N]
    x_map = X.astype(np.float32)
    y_map = Y.astype(np.float32)

    # pad to multiple of 8 for your 3-downsample UNet
    pad_h, pad_w = _pad_to_multiple(N, N, mult=8)
    Hpad, Wpad = N + pad_h, N + pad_w

    # ----------------- containers -----------------
    X_static_list = []
    YT_list = []
    YF_list = []
    case_meta = []

    # RNG for choosing modes (stable across runs)
    rng_mode = np.random.default_rng(2024)

    # ----------------- build cases -----------------
    for i in range(1, NUM_CASES + 1):

        # 1) BC dict (unchanged)
        bc = make_bc_case(i, T_bound=T_bound, only_lr=only_lr_vary, all_const=All_side_const_temp_boundary)

        # 2) Choose Q mode for this case
        r = float(rng_mode.random())
        if r < p_gp:
            q_mode = "gp"
        elif r < p_gp + p_bat:
            q_mode = "battery"
        else:
            q_mode = "hybrid"

        # 3) Build Q + bat_mask depending on mode
        # Always produce:
        #   Q: [N,N] float32
        #   bat_mask: [N,N] float32 (0/1)
        bat_mask = np.zeros((N, N), dtype=np.float32)

        # ---- GP component (if used) ----
        Q_gp = None
        if q_mode in ("gp", "hybrid"):
            genQ = HeatSource2DRBFPosterior(
                grid_size=N, length=1.0, length_scale=Q_length_scale,
                sigma=Q_sigma, jitter=1e-6, seed=10_000 + i
            )
            Q_dimless = genQ.sample_posterior(
                enforce_zero_mean=False,
                force_sign="positive",
                target_mean=1.0,
                keep_boundary_zero=False
            )
            Q_gp = _ensure_full_Q(q_scale * Q_dimless, N).astype(np.float32)  # [N,N]

        # ---- Battery component (if used) ----
        Q_bat = None
        bats = None
        if q_mode in ("battery", "hybrid"):
            rng_b = np.random.default_rng(50_000 + i)

            # Use your existing sampler names (cells-based)
            bats = sample_batteries_hw(
                rng_b, nx=N, ny=N,
                Nb_range=BAT_NB_RANGE,
                size_range=BAT_SIZE_RANGE,
                q_range=BAT_Q_RANGE,
                margin=BAT_MARGIN,
                allow_overlap=False
            )
            Q_bat, bat_mask = render_battery_Q_and_mask(
                nx=N, ny=N, batteries=bats,
                base=BAT_BASE,
                soft_edges=BAT_SOFT_EDGES,
                blur_iters=BAT_BLUR_ITERS
            )

        # ---- Combine final Q ----
        if q_mode == "gp":
            Q = Q_gp
        elif q_mode == "battery":
            Q = Q_bat.astype(np.float32)
        else:
            # hybrid: weighted sum (keep everything positive)
            w_gp  = float(np.random.default_rng(60_000 + i).uniform(*HYB_W_GP_RANGE))
            w_bat = float(np.random.default_rng(70_000 + i).uniform(*HYB_W_BAT_RANGE))
            Q = (w_gp * Q_gp + w_bat * Q_bat).astype(np.float32)

        # 4) Simulate PCM (battery_mask only matters if q_mode has battery)
        Ts, Fs, ts, Xsim, Ysim, info = simulate_pcm_2d_with_source(
            Q_Wm3=Q, nx=N, ny=N, Lx=Lx, Ly=Ly,
            rho=rho, cp=cp, k=k, L_lat=L_lat,
            T_init=T_init, Tm=T_m,
            t_end=t_end, cfl=cfl, save_times=save_times, bc=bc,
            battery_mask=bat_mask if q_mode in ("battery", "hybrid") else None
        )
        TT = np.asarray(Ts, dtype=np.float32)  # [T, N, N]
        FF = np.asarray(Fs, dtype=np.float32)  # [T, N, N]

        # 5) Build static channels (images) — keep names consistent
        bc_maps = make_bc_maps(bc, X=x_map, Y=y_map, Lx=Lx, Ly=Ly)  # 8 maps [N,N] each
        ones = np.ones((N, N), dtype=np.float32)

        # IMPORTANT: keep a single Cin for all cases by always including bat_mask.
        # Channels (new but consistent variables): [Q, bat_mask, x, y, 8 BC maps, ones]
        channels = [Q, bat_mask, x_map, y_map] + bc_maps + [ones]   # 1+1+2 + 8 + 1 = 13
        X_static = np.stack(channels, axis=0).astype(np.float32)     # [Cin, N, N]
        X_static = _apply_pad(X_static, pad_h, pad_w, mode="constant", constant_values=0.0)  # [Cin,Hpad,Wpad]

        # 6) Pad targets too (so UNet output aligns)
        YT = _apply_pad(TT, pad_h, pad_w, mode="constant", constant_values=0.0)  # [T,Hpad,Wpad]
        YF = _apply_pad(FF, pad_h, pad_w, mode="constant", constant_values=0.0)

        # 7) Append
        X_static_list.append(X_static)
        YT_list.append(YT)
        YF_list.append(YF)

        # keep your meta pattern but include q_mode
        case_meta.append({"case_id": i, "bc": bc, "q_mode": q_mode, "batteries": bats})

        # 8) Keep your logging / stats (unchanged call signature)
        _ = write_case_stats(
            RUN_DIR=RUN_DIR, case_id=i, Q=Q, TT=TT, FF=FF,
            save_times=save_times, bc=bc
        )

        # extra sanity print (short)
        if q_mode in ("battery", "hybrid"):
            bat_pixels = int((bat_mask > 0.5).sum())
        else:
            bat_pixels = 0
        print(f"[case {i:>2d}/{NUM_CASES}] mode={q_mode:<7} bat_pix={bat_pixels:<5d} "
              f"X_static={X_static.shape}  YT={YT.shape}  YF={YF.shape}")

    # ----------------- stack final arrays -----------------
    X_static = np.stack(X_static_list, axis=0)  # [C, Cin, Hpad, Wpad]
    Y_T      = np.stack(YT_list, axis=0)        # [C, T,   Hpad, Wpad]
    Y_f      = np.stack(YF_list, axis=0)        # [C, T,   Hpad, Wpad]
    case_ids = np.arange(NUM_CASES, dtype=np.int64)

    meta = dict(
        format="udeeponet_grid",
        N=N, Lx=Lx, Ly=Ly,
        Hpad=Hpad, Wpad=Wpad,
        pad_h=int(pad_h), pad_w=int(pad_w),
        Cin=int(X_static.shape[1]),
        times=times.tolist(),
        note="X_static channels: [Q,bat_mask,x,y,left_val,left_mask,right_val,right_mask,bottom_val,bottom_mask,top_val,top_mask,ones]",
        q_mode_probs={"gp": p_gp, "battery": p_bat, "hybrid": p_hyb},
        battery_params=dict(
            Nb_range=BAT_NB_RANGE,
            size_range=BAT_SIZE_RANGE,
            q_range=BAT_Q_RANGE,
            margin=BAT_MARGIN,
            soft_edges=BAT_SOFT_EDGES,
            blur_iters=BAT_BLUR_ITERS,
            base=BAT_BASE,
        ),
        hybrid_params=dict(
            w_gp_range=HYB_W_GP_RANGE,
            w_bat_range=HYB_W_BAT_RANGE
        )
    )

    # ----------------- splits (case-wise) -----------------
    rng = np.random.default_rng(2024)
    all_cases = np.arange(NUM_CASES, dtype=int)
    rng.shuffle(all_cases)
    n_train = int(round(NUM_CASES * TRAIN_FRAC))
    n_val   = int(round(NUM_CASES * VAL_FRAC))
    split = {
        "train": all_cases[:n_train].tolist(),
        "val":   all_cases[n_train:n_train+n_val].tolist(),
        "test":  all_cases[n_train+n_val:].tolist()
    }

    # ----------------- save npz -----------------
    out_path = RUN_DIR / "udeeponet_dataset.npz"
    #np.savez_compressed(
    #    out_path,
    #    X_static=X_static,
    #    Y_T=Y_T,
    #    Y_f=Y_f,
    #    times=times,
    #    case_ids=case_ids,
    #    #meta=json.dumps(meta),
    #    #split=json.dumps(split),
    #    #case_meta=json.dumps(case_meta)
    #    meta=np.array(meta, dtype=object),
    #)
    #with open(RUN_DIR / "cases_bc.json", "w") as f:
    #    json.dump(case_meta, f, indent=2)
    #with open(RUN_DIR / "splits.json", "w") as f:
    #    json.dump(split, f, indent=2)

    np.savez_compressed(
    out_path,
    X_static=X_static,
    Y_T=Y_T,
    Y_f=Y_f,
    times=times,
    case_ids=case_ids,
    meta=np.array(meta, dtype=object),          # ✅ key change (robust old-style)
    split=np.array(split, dtype=object),        # optional but nice
    case_meta=np.array(case_meta, dtype=object) # optional but nice
    )

    
    with open(RUN_DIR / "meta.json", "w") as f:
        json.dump(meta, f, indent=2)

    with open(RUN_DIR / "splits.json", "w") as f:
        json.dump(split, f, indent=2)

    # IMPORTANT: cases_bc.json should be *only* BCs, not whole case_meta list
    cases_bc = {str(d["case_id"]): d["bc"] for d in case_meta}
    with open(RUN_DIR / "cases_bc.json", "w") as f:
        json.dump(cases_bc, f, indent=2)

    # (optional) also keep full case_meta if you want batteries/q_mode saved
    with open(RUN_DIR / "case_meta.json", "w") as f:
        json.dump(case_meta, f, indent=2)


    print("────────────────────────────────────────────────────────")
    print("DONE")
    print("X_static:", X_static.shape)
    print("Y_T:", Y_T.shape)
    print("Y_f:", Y_f.shape)
    print("times:", times.tolist())
    print("Saved dataset to:", str(out_path.resolve()))
    print("────────────────────────────────────────────────────────")

    return out_path      

In [ ]:
'''# ============================================================
# DATA GENERATOR (generalized Q modes: gp / battery / hybrid)
# Saves: udeeponet_dataset.npz (arrays only) + meta.json + splits.json + cases_bc.json
# ============================================================

def build_udeeponet_dataset():
    """
    U-DeepONet dataset (grid-to-grid):
      X_static: [C, Cin, Hpad, Wpad]
      times:    [T]
      Y_T:      [C, T, Hpad, Wpad]
      Y_f:      [C, T, Hpad, Wpad]
    """

    # ----------------- pull globals (keep your style) -----------------
    N        = int(globals()["N"])
    Lx       = float(globals()["Lx"])
    Ly       = float(globals()["Ly"])
    rho      = float(globals()["rho"])
    cp       = float(globals()["cp"])
    k        = float(globals()["k"])
    L_lat    = float(globals()["L_lat"])
    T_m      = float(globals().get("T_m", globals().get("Tm")))
    T_init   = float(globals()["T_init"])
    T_bound  = float(globals()["T_bound"])
    t_end    = float(globals()["t_end"])
    save_times = tuple(globals()["save_times"])
    cfl      = float(globals()["cfl"])

    # GP / Q field params (already in your notebook)
    q_scale        = float(globals()["q_scale"])
    Q_length_scale = float(globals()["Q_length_scale"])
    Q_sigma        = float(globals()["Q_sigma"])

    # dataset sizes
    NUM_CASES      = int(globals()["NUM_CASES"])
    TRAIN_FRAC     = float(globals()["TRAIN_FRAC"])
    VAL_FRAC       = float(globals()["VAL_FRAC"])

    # boundary variety
    only_lr_vary   = bool(globals()["only_lr_vary"])
    All_side_const_temp_boundary = bool(globals()["All_side_const_temp_boundary"])

    # output dir
    RUN_DIR = Path(globals()["RUN_DIR"])
    RUN_DIR.mkdir(parents=True, exist_ok=True)

    # ----------------- NEW: Q mode mixing knobs (optional globals) -----------------
    # If you don't define these globals, defaults below will be used.
    # Probabilities must sum to 1.0
    Q_MODE_PROBS = globals().get("Q_MODE_PROBS", {"gp": 0.45, "battery": 0.35, "hybrid": 0.20})
    p_gp     = float(Q_MODE_PROBS.get("gp", 0.45))
    p_bat    = float(Q_MODE_PROBS.get("battery", 0.35))
    p_hyb    = float(Q_MODE_PROBS.get("hybrid", 0.20))
    s = p_gp + p_bat + p_hyb
    if abs(s - 1.0) > 1e-6:
        p_gp, p_bat, p_hyb = p_gp/s, p_bat/s, p_hyb/s

    # battery params (optional globals)
    BAT_NB_RANGE   = globals().get("BAT_NB_RANGE", (2, 5))
    BAT_SIZE_RANGE = globals().get("BAT_SIZE_RANGE", (5, 16))          # in cells
    BAT_Q_RANGE    = globals().get("BAT_Q_RANGE", (1.0e6, 1.2e6))
    BAT_MARGIN     = int(globals().get("BAT_MARGIN", 3))
    BAT_SOFT_EDGES = bool(globals().get("BAT_SOFT_EDGES", True))
    BAT_BLUR_ITERS = int(globals().get("BAT_BLUR_ITERS", 1))
    BAT_BASE       = float(globals().get("BAT_BASE", 4.5))             # background

    # hybrid mixing weights (optional globals)
    HYB_W_GP_RANGE  = globals().get("HYB_W_GP_RANGE",  (0.4, 0.7))
    HYB_W_BAT_RANGE = globals().get("HYB_W_BAT_RANGE", (0.3, 0.6))

    # ----------------- header -----------------
    times = np.array(sorted(save_times), dtype=np.float32)
    T = len(times)
    print("────────────────────────────────────────────────────────")
    print("Building U-DeepONet dataset (grid-to-grid)")
    print(f"N={N} | times={times.tolist()} | NUM_CASES={NUM_CASES}")
    print(f"Q modes probs: gp={p_gp:.2f}, battery={p_bat:.2f}, hybrid={p_hyb:.2f}")
    print(f"Output dir: {RUN_DIR.resolve()}")
    print("────────────────────────────────────────────────────────")

    # ----------------- precompute coordinate maps -----------------
    x = np.linspace(0.0, Lx, N, dtype=np.float32)
    y = np.linspace(0.0, Ly, N, dtype=np.float32)
    X, Y = np.meshgrid(x, y, indexing="ij")  # [N,N]
    x_map = X.astype(np.float32)
    y_map = Y.astype(np.float32)

    # pad to multiple of 8 for your 3-downsample UNet
    pad_h, pad_w = _pad_to_multiple(N, N, mult=8)
    Hpad, Wpad = N + pad_h, N + pad_w

    # ----------------- containers -----------------
    X_static_list = []
    YT_list = []
    YF_list = []
    case_meta = []

    # RNG for choosing modes (stable across runs)
    rng_mode = np.random.default_rng(2024)

    # ----------------- build cases -----------------
    for i in range(1, NUM_CASES + 1):

        # 1) BC dict (unchanged)
        bc = make_bc_case(i, T_bound=T_bound, only_lr=only_lr_vary, all_const=All_side_const_temp_boundary)

        # 2) Choose Q mode for this case
        r = float(rng_mode.random())
        if r < p_gp:
            q_mode = "gp"
        elif r < p_gp + p_bat:
            q_mode = "battery"
        else:
            q_mode = "hybrid"

        # 3) Build Q + bat_mask depending on mode
        # Always produce:
        #   Q: [N,N] float32
        #   bat_mask: [N,N] float32 (0/1)
        bat_mask = np.zeros((N, N), dtype=np.float32)

        # ---- GP component (if used) ----
        Q_gp = None
        if q_mode in ("gp", "hybrid"):
            genQ = HeatSource2DRBFPosterior(
                grid_size=N, length=1.0, length_scale=Q_length_scale,
                sigma=Q_sigma, jitter=1e-6, seed=10_000 + i
            )
            Q_dimless = genQ.sample_posterior(
                enforce_zero_mean=False,
                force_sign="positive",
                target_mean=1.0,
                keep_boundary_zero=False
            )
            Q_gp = _ensure_full_Q(q_scale * Q_dimless, N).astype(np.float32)  # [N,N]

        # ---- Battery component (if used) ----
        Q_bat = None
        bats = None
        if q_mode in ("battery", "hybrid"):
            rng_b = np.random.default_rng(50_000 + i)

            bats = sample_batteries_hw(
                rng_b, nx=N, ny=N,
                Nb_range=BAT_NB_RANGE,
                size_range=BAT_SIZE_RANGE,
                q_range=BAT_Q_RANGE,
                margin=BAT_MARGIN,
                allow_overlap=False
            )
            Q_bat, bat_mask = render_battery_Q_and_mask(
                nx=N, ny=N, batteries=bats,
                base=BAT_BASE,
                soft_edges=BAT_SOFT_EDGES,
                blur_iters=BAT_BLUR_ITERS
            )
            Q_bat = Q_bat.astype(np.float32)
            bat_mask = bat_mask.astype(np.float32)

        # ---- Combine final Q ----
        if q_mode == "gp":
            Q = Q_gp
        elif q_mode == "battery":
            Q = Q_bat
        else:
            w_gp  = float(np.random.default_rng(60_000 + i).uniform(*HYB_W_GP_RANGE))
            w_bat = float(np.random.default_rng(70_000 + i).uniform(*HYB_W_BAT_RANGE))
            Q = (w_gp * Q_gp + w_bat * Q_bat).astype(np.float32)

        # 4) Simulate PCM (battery_mask only matters if q_mode has battery)
        Ts, Fs, ts, Xsim, Ysim, info = simulate_pcm_2d_with_source(
            Q_Wm3=Q, nx=N, ny=N, Lx=Lx, Ly=Ly,
            rho=rho, cp=cp, k=k, L_lat=L_lat,
            T_init=T_init, Tm=T_m,
            t_end=t_end, cfl=cfl, save_times=save_times, bc=bc,
            battery_mask=bat_mask if q_mode in ("battery", "hybrid") else None
        )
        TT = np.asarray(Ts, dtype=np.float32)  # [T, N, N]
        FF = np.asarray(Fs, dtype=np.float32)  # [T, N, N]

        # 5) Build static channels (images)
        bc_maps = make_bc_maps(bc, X=x_map, Y=y_map, Lx=Lx, Ly=Ly)  # 8 maps [N,N]
        ones = np.ones((N, N), dtype=np.float32)

        # Channels: [Q, bat_mask, x, y, 8 BC maps, ones] => 13
        channels = [Q, bat_mask, x_map, y_map] + bc_maps + [ones]
        X_static = np.stack(channels, axis=0).astype(np.float32)     # [Cin, N, N]
        X_static = _apply_pad(X_static, pad_h, pad_w, mode="constant", constant_values=0.0)  # [Cin,Hpad,Wpad]

        # 6) Pad targets too
        YT = _apply_pad(TT, pad_h, pad_w, mode="constant", constant_values=0.0)  # [T,Hpad,Wpad]
        YF = _apply_pad(FF, pad_h, pad_w, mode="constant", constant_values=0.0)

        # 7) Append
        X_static_list.append(X_static)
        YT_list.append(YT)
        YF_list.append(YF)

        # 8) Meta per case
        case_meta.append({"case_id": i, "bc": bc, "q_mode": q_mode, "batteries": bats})

        # 9) Your stats writer (unchanged signature)
        _ = write_case_stats(
            RUN_DIR=RUN_DIR, case_id=i, Q=Q, TT=TT, FF=FF,
            save_times=save_times, bc=bc
        )

        # 10) short sanity print
        if q_mode in ("battery", "hybrid"):
            bat_pixels = int((bat_mask > 0.5).sum())
        else:
            bat_pixels = 0
        print(f"[case {i:>2d}/{NUM_CASES}] mode={q_mode:<7} bat_pix={bat_pixels:<5d} "
              f"X_static={X_static.shape}  YT={YT.shape}  YF={YF.shape}")

    # ----------------- stack final arrays -----------------
    X_static = np.stack(X_static_list, axis=0)  # [C, Cin, Hpad, Wpad]
    Y_T      = np.stack(YT_list, axis=0)        # [C, T,   Hpad, Wpad]
    Y_f      = np.stack(YF_list, axis=0)        # [C, T,   Hpad, Wpad]
    case_ids = np.arange(NUM_CASES, dtype=np.int64)

    meta = dict(
        format="udeeponet_grid",
        N=N, Lx=Lx, Ly=Ly,
        Hpad=Hpad, Wpad=Wpad,
        pad_h=int(pad_h), pad_w=int(pad_w),
        Cin=int(X_static.shape[1]),
        times=times.tolist(),
        note="X_static channels: [Q,bat_mask,x,y,left_val,left_mask,right_val,right_mask,bottom_val,bottom_mask,top_val,top_mask,ones]",
        q_mode_probs={"gp": p_gp, "battery": p_bat, "hybrid": p_hyb},
        battery_params=dict(
            Nb_range=list(BAT_NB_RANGE),
            size_range=list(BAT_SIZE_RANGE),
            q_range=list(BAT_Q_RANGE),
            margin=int(BAT_MARGIN),
            soft_edges=bool(BAT_SOFT_EDGES),
            blur_iters=int(BAT_BLUR_ITERS),
            base=float(BAT_BASE),
        ),
        hybrid_params=dict(
            w_gp_range=list(HYB_W_GP_RANGE),
            w_bat_range=list(HYB_W_BAT_RANGE),
        ),
    )

    # ----------------- splits (case-wise) -----------------
    rng = np.random.default_rng(2024)
    all_cases = np.arange(NUM_CASES, dtype=int)
    rng.shuffle(all_cases)
    n_train = int(round(NUM_CASES * TRAIN_FRAC))
    n_val   = int(round(NUM_CASES * VAL_FRAC))
    split = {
        "train": all_cases[:n_train].tolist(),
        "val":   all_cases[n_train:n_train+n_val].tolist(),
        "test":  all_cases[n_train+n_val:].tolist()
    }

    # ----------------- save: npz arrays only + separate json files -----------------
    out_path = RUN_DIR / "udeeponet_dataset.npz"
    np.savez_compressed(
        out_path,
        X_static=X_static,
        Y_T=Y_T,
        Y_f=Y_f,
        times=times,
        case_ids=case_ids,
        meta=meta
    )

    with open(RUN_DIR / "meta.json", "w") as f:
        json.dump(meta, f, indent=2)

    with open(RUN_DIR / "cases_bc.json", "w") as f:
        json.dump(case_meta, f, indent=2)

    with open(RUN_DIR / "splits.json", "w") as f:
        json.dump(split, f, indent=2)

    print("────────────────────────────────────────────────────────")
    print("DONE")
    print("X_static:", X_static.shape)
    print("Y_T:", Y_T.shape)
    print("Y_f:", Y_f.shape)
    print("times:", times.tolist())
    print("Saved dataset to:", str(out_path.resolve()))
    print("Also saved: meta.json, cases_bc.json, splits.json")
    print("────────────────────────────────────────────────────────")

    return out_path'''


In [ ]:
build_udeeponet_dataset()

In [ ]:
data = np.load(RUN_DIR / "udeeponet_dataset.npz", allow_pickle=True)

X_static = data["X_static"]   # [C, Cin, H, W]
times    = data["times"]      # [T]
Y_T      = data["Y_T"]        # [C, T, H, W]
Y_f      = data["Y_f"]        # [C, T, H, W]
meta     = data["meta"].item()

print("X_static:", X_static.shape)
print("Y_T:", Y_T.shape)
print("Y_f:", Y_f.shape)
print("times:", times)

H,W=X_static.shape[-2:]


In [ ]:
case = 0

Q = X_static[case, 0]  # [H,W]

plt.figure(figsize=(5,4))
plt.imshow(Q, origin="lower", cmap="inferno")
plt.colorbar(label="Q (W/m³)")
plt.title(f"Heat source Q – case {case}")
plt.tight_layout()
plt.show()


In [ ]:
case = 0

for ti, t in enumerate(times):
    plt.figure(figsize=(5,4))
    plt.imshow(Y_T[case, ti], origin="lower", cmap="inferno")
    plt.colorbar(label="Temperature")
    plt.title(f"T(x,y) | case {case} | t = {t}")
    plt.tight_layout()
    plt.show()


In [ ]:
case = 0

for ti, t in enumerate(times):
    plt.figure(figsize=(5,4))
    plt.imshow(Y_f[case, ti], origin="lower", cmap="viridis", vmin=0, vmax=1)
    plt.colorbar(label="Liquid fraction")
    plt.title(f"f(x,y) | case {case} | t = {t}")
    plt.tight_layout()
    plt.show()


In [ ]:
def plot_boundary_1d(case_id):
    X = X_static[case_id]

    left_val   = X[3]; left_mask   = X[4]
    right_val  = X[5]; right_mask  = X[6]
    bottom_val = X[7]; bottom_mask = X[8]
    top_val    = X[9]; top_mask    = X[10]

    fig, axs = plt.subplots(2, 2, figsize=(10, 6))

    # Left boundary (varies along y)
    y = np.arange(H)
    axs[0,0].plot(y[left_mask[:,0] > 0], left_val[left_mask[:,0] > 0, 0])
    axs[0,0].set_title("Left boundary")
    axs[0,0].set_xlabel("y-index")
    axs[0,0].set_ylabel("Temperature")

    # Right boundary
    axs[0,1].plot(y[right_mask[:,-1] > 0], right_val[right_mask[:,-1] > 0, -1])
    axs[0,1].set_title("Right boundary")
    axs[0,1].set_xlabel("y-index")

    # Bottom boundary (varies along x)
    x = np.arange(W)
    axs[1,0].plot(x[bottom_mask[0,:] > 0], bottom_val[0, bottom_mask[0,:] > 0])
    axs[1,0].set_title("Bottom boundary")
    axs[1,0].set_xlabel("x-index")
    axs[1,0].set_ylabel("Temperature")

    # Top boundary
    axs[1,1].plot(x[top_mask[-1,:] > 0], top_val[-1, top_mask[-1,:] > 0])
    axs[1,1].set_title("Top boundary")
    axs[1,1].set_xlabel("x-index")

    fig.suptitle(f"Boundary conditions – case {case_id}")
    plt.tight_layout()
    plt.show()

In [ ]:
from datetime import datetime

def _ensure_dir(p: Path):
    p.mkdir(parents=True, exist_ok=True)
    return p


def _get_channel_map(meta):
    """
    Returns indices for channels based on meta['note'] if available,
    else falls back to the known default ordering:
    [Q,bat_mask,x,y,left_val,left_mask,right_val,right_mask,bottom_val,bottom_mask,top_val,top_mask,ones]
    """
    note = meta.get("note", "")
    if "X_static channels" in note and "[" in note and "]" in note:
        # parse inside [...]
        inside = note.split("[", 1)[1].rsplit("]", 1)[0]
        names = [s.strip() for s in inside.split(",")]
        return {name: i for i, name in enumerate(names)}

    # fallback
    names = ["Q","bat_mask","x","y",
             "left_val","left_mask",
             "right_val","right_mask",
             "bottom_val","bottom_mask",
             "top_val","top_mask",
             "ones"]
    return {name: i for i, name in enumerate(names)}


def _plot_boundary_profiles_from_Xstatic(X_case, meta, case_id, outpath: Path):
    cmap = _get_channel_map(meta)

    left_val, left_mask     = X_case[cmap["left_val"]],   X_case[cmap["left_mask"]]
    right_val, right_mask   = X_case[cmap["right_val"]],  X_case[cmap["right_mask"]]
    bottom_val, bottom_mask = X_case[cmap["bottom_val"]], X_case[cmap["bottom_mask"]]
    top_val, top_mask       = X_case[cmap["top_val"]],    X_case[cmap["top_mask"]]

    H, W = X_case.shape[-2:]
    Lx, Ly = float(meta["Lx"]), float(meta["Ly"])

    x_phys = np.linspace(0, Lx, H)   # axis-0 = x
    y_phys = np.linspace(0, Ly, W)   # axis-1 = y

    fig, axs = plt.subplots(2, 2, figsize=(10, 7), constrained_layout=True)

    # LEFT: i=0 row, varies along y (j)
    m = left_mask[0, :] > 0.5
    axs[0,0].plot(y_phys[m], left_val[0, m], lw=2)
    axs[0,0].set_title("Left (x=0)")
    axs[0,0].set_xlabel("y [m]"); axs[0,0].set_ylabel("T [K]")
    axs[0,0].grid(alpha=0.25)

    # RIGHT: i=-1 row, varies along y
    m = right_mask[-1, :] > 0.5
    axs[0,1].plot(y_phys[m], right_val[-1, m], lw=2)
    axs[0,1].set_title("Right (x=Lx)")
    axs[0,1].set_xlabel("y [m]"); axs[0,1].set_ylabel("T [K]")
    axs[0,1].grid(alpha=0.25)

    # BOTTOM: j=0 col, varies along x
    m = bottom_mask[:, 0] > 0.5
    axs[1,0].plot(x_phys[m], bottom_val[m, 0], lw=2)
    axs[1,0].set_title("Bottom (y=0)")
    axs[1,0].set_xlabel("x [m]"); axs[1,0].set_ylabel("T [K]")
    axs[1,0].grid(alpha=0.25)

    # TOP: j=-1 col, varies along x
    m = top_mask[:, -1] > 0.5
    axs[1,1].plot(x_phys[m], top_val[m, -1], lw=2)
    axs[1,1].set_title("Top (y=Ly)")
    axs[1,1].set_xlabel("x [m]"); axs[1,1].set_ylabel("T [K]")
    axs[1,1].grid(alpha=0.25)

    fig.suptitle(f"Case {case_id:03d} boundary profiles")
    fig.savefig(outpath, dpi=200)
    plt.close(fig)

def _plot_Q_T_F_grid(X_case, YT_case, YF_case, times, meta, case_id, outpath: Path,
                     t_indices=None, show_battery_outline=True):
    """
    Correct plotting for arrays stored as [x,y] (indexing='ij').
    Uses .T in imshow so x is horizontal and y is vertical.
    """
    cmap = _get_channel_map(meta)

    Q = X_case[cmap["Q"]]         # [H,W] = [x,y]
    bat_mask = X_case[cmap["bat_mask"]] if "bat_mask" in cmap else None

    H, W = Q.shape
    Lx, Ly = float(meta["Lx"]), float(meta["Ly"])
    extent = [0, Lx, 0, Ly]

    Tn = len(times)
    if t_indices is None:
        K = min(9, Tn)
        t_indices = np.linspace(0, Tn-1, K).round().astype(int).tolist()
    K = len(t_indices)

    fig = plt.figure(figsize=(2.9*K, 2.6*3), dpi=160)

    # coordinates for contours (must match transposed image)
    x = np.linspace(0, Lx, H)   # axis-0 length
    y = np.linspace(0, Ly, W)   # axis-1 length
    Xg, Yg = np.meshgrid(x, y, indexing="xy")  # shapes [W,H] to match .T plots

    # --- Row 1: Q
    for j in range(K):
        ax = plt.subplot(3, K, 1 + j)
        im = ax.imshow(Q.T, origin="lower", extent=extent, aspect="equal")
        ax.set_title("Heat source Q")
        ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.02, label="Q [W/m³]")

        if show_battery_outline and bat_mask is not None:
            ax.contour(Xg, Yg, bat_mask.T, levels=[0.5], linewidths=1.0)

    # --- Row 2: Temperature
    for j, ti in enumerate(t_indices):
        ax = plt.subplot(3, K, 1*K + 1 + j)
        im = ax.imshow(YT_case[ti].T, origin="lower", extent=extent, aspect="equal")
        ax.set_title(f"T at t={times[ti]:.1f} s")
        ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.02, label="T [K]")

        if show_battery_outline and bat_mask is not None:
            ax.contour(Xg, Yg, bat_mask.T, levels=[0.5], linewidths=1.0)

    # --- Row 3: Liquid fraction
    for j, ti in enumerate(t_indices):
        ax = plt.subplot(3, K, 2*K + 1 + j)
        im = ax.imshow(YF_case[ti].T, origin="lower", extent=extent, aspect="equal",
                       vmin=0.0, vmax=1.0)
        ax.set_title(f"Liquid fraction at t={times[ti]:.1f} s")
        ax.set_xlabel("x [m]"); ax.set_ylabel("y [m]")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.02, label="f [-]")

        if show_battery_outline and bat_mask is not None:
            ax.contour(Xg, Yg, bat_mask.T, levels=[0.5], linewidths=1.0)

    plt.tight_layout()
    fig.savefig(outpath, dpi=200)
    plt.close(fig)



def _time_stats(YT_case, YF_case):
    """
    YT_case: [T,H,W]
    YF_case: [T,H,W]
    returns dict arrays (min/mean/max/var vs time)
    """
    T = YT_case.shape[0]
    stats = {
        "T_min": np.zeros(T), "T_mean": np.zeros(T), "T_max": np.zeros(T), "T_var": np.zeros(T),
        "f_min": np.zeros(T), "f_mean": np.zeros(T), "f_max": np.zeros(T), "f_var": np.zeros(T),
    }
    for ti in range(T):
        A = YT_case[ti]
        B = YF_case[ti]
        stats["T_min"][ti]  = float(A.min())
        stats["T_mean"][ti] = float(A.mean())
        stats["T_max"][ti]  = float(A.max())
        stats["T_var"][ti]  = float(A.var())
        stats["f_min"][ti]  = float(B.min())
        stats["f_mean"][ti] = float(B.mean())
        stats["f_max"][ti]  = float(B.max())
        stats["f_var"][ti]  = float(B.var())
    return stats

def _plot_time_stats(times, stats, case_id, outpath: Path):
    fig, axs = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

    axs[0].plot(times, stats["T_min"],  marker="o", label="min")
    axs[0].plot(times, stats["T_mean"], marker="o", label="mean")
    axs[0].plot(times, stats["T_max"],  marker="o", label="max")
    axs[0].set_title("Temperature stats vs time")
    axs[0].set_ylabel("T [K]")
    axs[0].grid(True, alpha=0.3)
    axs[0].legend()

    axs[1].plot(times, stats["f_min"],  marker="o", label="min")
    axs[1].plot(times, stats["f_mean"], marker="o", label="mean")
    axs[1].plot(times, stats["f_max"],  marker="o", label="max")
    axs[1].set_title("Liquid fraction stats vs time")
    axs[1].set_xlabel("time (s)")
    axs[1].set_ylabel("f [-]")
    axs[1].grid(True, alpha=0.3)
    axs[1].legend()

    fig.suptitle(f"Case {case_id:03d} time stats")
    plt.tight_layout()
    fig.savefig(outpath, dpi=200)
    plt.close(fig)

def _plot_variance_stats(times, stats, case_id, outpath: Path):
    fig = plt.figure(figsize=(10, 5))
    plt.plot(times, stats["T_var"], marker="o", label="T variance")
    plt.plot(times, stats["f_var"], marker="o", label="f variance")
    plt.title("Variance vs time")
    plt.xlabel("time (s)")
    plt.ylabel("Variance")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    fig.savefig(outpath, dpi=200)
    plt.close(fig)

def save_dataset_run_with_all_plots(base_out_dir: Path, dataset_npz_path: Path,
                                    run_name=None):
    if run_name is None:
        run_name = "dataset_run_" + datetime.now().strftime("%Y%m%d-%H%M%S")

    RUN = _ensure_dir(base_out_dir / run_name)
    plots_dir = _ensure_dir(RUN / "plots")

    data = np.load(dataset_npz_path, allow_pickle=True)

    X_static = data["X_static"]
    times = data["times"].astype(float)
    Y_T = data["Y_T"]
    Y_f = data["Y_f"]

    # ---- FIX: meta may be JSON string ----
    meta_raw = data["meta"].item()
    meta = json.loads(meta_raw) if isinstance(meta_raw, (str, bytes)) else meta_raw

    # save a copy (keep same structure)
    np.savez_compressed(
        RUN / "udeeponet_dataset.npz",
        X_static=X_static, times=times, Y_T=Y_T, Y_f=Y_f,
        case_ids=data["case_ids"],
        meta=json.dumps(meta)
    )

    parent = dataset_npz_path.parent
    for fname in ["splits.json", "cases_bc.json"]:
        src = parent / fname
        if src.exists():
            (RUN / fname).write_text(src.read_text())

    C = X_static.shape[0]

    global_rows = []
    for c in range(C):
        Xc  = X_static[c]
        YTc = Y_T[c]
        YFc = Y_f[c]

        _plot_boundary_profiles_from_Xstatic(
            Xc, meta, c+1, plots_dir / f"case_{c+1:03d}_boundary_profiles.png"
        )
        _plot_Q_T_F_grid(
            Xc, YTc, YFc, times, meta, c+1,
            plots_dir / f"case_{c+1:03d}_temp_and_liquid_profiles.png"
        )

        stats = _time_stats(YTc, YFc)
        _plot_time_stats(times, stats, c+1, plots_dir / f"case_{c+1:03d}_time_stats.png")
        _plot_variance_stats(times, stats, c+1, plots_dir / f"case_{c+1:03d}_variance_stats.png")

        global_rows.append({
            "case": c+1,
            "Q_min": float(Xc[_get_channel_map(meta)["Q"]].min()),
            "Q_mean": float(Xc[_get_channel_map(meta)["Q"]].mean()),
            "Q_max": float(Xc[_get_channel_map(meta)["Q"]].max()),
            "T_min_all": float(YTc.min()),
            "T_mean_all": float(YTc.mean()),
            "T_max_all": float(YTc.max()),
            "f_min_all": float(YFc.min()),
            "f_mean_all": float(YFc.mean()),
            "f_max_all": float(YFc.max()),
        })

        if (c+1) % 10 == 0 or (c+1) == C:
            print(f"Saved plots + stats for case {c+1}/{C}")

    csv_path = RUN / "global_case_summary.csv"
    with open(csv_path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(global_rows[0].keys()))
        w.writeheader()
        w.writerows(global_rows)

    with open(RUN / "meta.json", "w") as f:
        json.dump(meta, f, indent=2)

    print("─────────────────────────────────────────────")
    print("Dataset run saved at:", RUN)
    print("Plots folder:", plots_dir)
    print("Global summary:", csv_path)
    print("Dataset file:", RUN / "udeeponet_dataset.npz")
    print("─────────────────────────────────────────────")
    return RUN


In [ ]:
base_out = Path(".")  # or wherever you want dataset_run folders
dataset_path = RUN_DIR / "udeeponet_dataset.npz"

run_folder = save_dataset_run_with_all_plots(
    base_out_dir=base_out,
    dataset_npz_path=dataset_path,
)


DATA ANALYSIS

In [ ]:
temp=r"runs/dataset_run_20251120-131248/deeponet_temp_dataset.npz"
npz=np.load(temp,allow_pickle=True)
print("All keys in temp file are: ")
print(list(npz.keys()))

temp1=r"runs/dataset_run_20251120-131248/deeponet_frac_dataset.npz"
npz_frac=np.load(temp1,allow_pickle=True)
print("All keys in frac.npz file are: ")
print(list(npz_frac.keys()))

In [ ]:
# ==========================================
# Single-plots: per-case series for mean/min/max/var (T & f)
# ==========================================

#dataset_dir = r"runs/dataset_run_20251120-131248"
dataset_dir = r"runs/dataset_run_20251112-182232"
temp_npz = os.path.join(dataset_dir, "deeponet_temp_dataset.npz")
frac_npz = os.path.join(dataset_dir, "deeponet_frac_dataset.npz")

def _ensure(cond, msg):
    if not cond: raise RuntimeError(msg)

_ensure(os.path.isfile(temp_npz), f"Missing: {temp_npz}")
_ensure(os.path.isfile(frac_npz), f"Missing: {frac_npz}")

Tz = np.load(temp_npz, allow_pickle=True)
Fz = np.load(frac_npz, allow_pickle=True)

# Required keys in your point datasets:
for k in ["trunk_t","yT","case_ids"]:
    _ensure(k in Tz, f"Key '{k}' not found in {temp_npz}")

trunk_t_T = Tz["trunk_t"].squeeze().astype(float)
yT        = Tz["yT"].squeeze().astype(float)
case_T    = Tz["case_ids"].squeeze()

# Frac keys (support variants)
yF = Fz["yf"].squeeze().astype(float)

t_F = Fz["trunk_t"].squeeze().astype(float) if "trunk_t" in Fz else trunk_t_T
case_F = Fz["case_ids"].squeeze() if "case_ids" in Fz else case_T

# Helper: stable unique times (collapse float jitter)
def unique_times_stable(t, decimals=9):
    t = np.round(np.asarray(t, float), decimals=decimals)
    return np.unique(t)

# Per-case time series from (x,y,t) points: mean/min/max/var across spatial points at each t
def per_case_series(case_id, t_raw, y_scalar):
    m = (case_id == case_ids)   # will set case_ids before calls
    t = t_raw[m]; y = y_scalar[m]
    tr = np.round(t, 9)
    ut = np.unique(tr)
    K = len(ut)
    mean = np.empty(K); var = np.empty(K); mn = np.empty(K); mx = np.empty(K)
    for i, tt in enumerate(ut):
        yy = y[tr == tt]
        mean[i] = yy.mean()
        var[i]  = yy.var(ddof=1) if yy.size > 1 else 0.0
        mn[i]   = yy.min()
        mx[i]   = yy.max()
    return ut, dict(mean=mean, var=var, min=mn, max=mx)

# Build union time axes (temperature drives the reference axis)
U_times_T = unique_times_stable(trunk_t_T)
U_times_F = unique_times_stable(t_F)

# Cases present in both files
cases_T = pd.unique(case_T)
cases_F = pd.unique(case_F)
common_cases = np.intersect1d(cases_T, cases_F)

# Compute per-case series for T
case_ids = case_T  # used inside per_case_series
T_series = {}
for c in common_cases:
    ut, s = per_case_series(c, trunk_t_T, yT)
    T_series[int(c)] = (ut, s)

# Compute per-case series for f
case_ids = case_F
F_series = {}
for c in common_cases:
    ut, s = per_case_series(c, t_F, yF)
    F_series[int(c)] = (ut, s)

# Align any per-case series to a common axis (nearest exact match on save_times)
def align_to_axis(ut, arr, ref):
    out = np.full_like(ref, np.nan, dtype=float)
    pos = {float(v): i for i,v in enumerate(ut)}
    for i, tval in enumerate(ref):
        j = pos.get(float(tval), None)
        if j is not None: out[i] = arr[j]
    return out

# Build aligned matrices (n_cases x K) for each metric
def build_matrix(series_dict, ref_times, key):
    ids = sorted(series_dict.keys())
    M = np.empty((len(ids), len(ref_times)), dtype=float)
    M[:] = np.nan
    for r, cid in enumerate(ids):
        ut, s = series_dict[cid]
        M[r] = align_to_axis(ut, s[key], ref_times)
    return ids, M

idsT, T_mean = build_matrix(T_series, U_times_T, "mean")
_,    T_min  = build_matrix(T_series, U_times_T, "min")
_,    T_max  = build_matrix(T_series, U_times_T, "max")
_,    T_var  = build_matrix(T_series, U_times_T, "var")

idsF, F_mean = build_matrix(F_series, U_times_F, "mean")
_,    F_min  = build_matrix(F_series, U_times_F, "min")
_,    F_max  = build_matrix(F_series, U_times_F, "max")
_,    F_var  = build_matrix(F_series, U_times_F, "var")

# -------- plotting helpers (no huge legends) --------
def _multi_lines(ref_times, M, title, ylabel, legend=False):
    plt.figure(figsize=(8,5))
    for r in range(M.shape[0]):
        plt.plot(ref_times, M[r], lw=1)
    plt.title(title); plt.xlabel("time (s)"); plt.ylabel(ylabel)
    plt.grid(True, alpha=0.3)
    if legend:
        # Caution: 50 labels can clutter; use only if you truly want it
        labels = [f"Case {cid}" for cid in idsT]  # or idsF depending on matrix
        plt.legend(labels, fontsize=7, ncol=3)
    plt.tight_layout()

# -------- Temperature plots (all 50 cases on one plot each) --------
_multi_lines(U_times_T, T_mean, "Mean Temperature across cases", "T_mean (K)")
_multi_lines(U_times_T, T_max,  "Max Temperature across cases",  "T_max (K)")
_multi_lines(U_times_T, T_min,  "Min Temperature across cases",  "T_min (K)")
_multi_lines(U_times_T, T_var,  "Temperature Variance across cases", "Var(T)")

# -------- Liquid fraction plots --------
_multi_lines(U_times_F, F_mean, "Mean Liquid Fraction across cases", "f_mean (-)")
_multi_lines(U_times_F, F_max,  "Max Liquid Fraction across cases",  "f_max (-)")
_multi_lines(U_times_F, F_min,  "Min Liquid Fraction across cases",  "f_min (-)")
_multi_lines(U_times_F, F_var,  "Liquid Fraction Variance across cases", "Var(f)")


In [ ]:
# ==========================================
# Gaussian curves across cases for each snapshot
# ==========================================

import numpy as np
import matplotlib.pyplot as plt

def _gaussian(x, mu, sigma):
    """Standard normal PDF with mean mu, std sigma."""
    if sigma <= 0:
        # Degenerate case -> spike at mu (approx)
        sigma = 1e-6
    return (1.0 / (sigma * np.sqrt(2.0 * np.pi))) * np.exp(-0.5 * ((x - mu) / sigma) ** 2)


def plot_snapshot_gaussians(values_matrix, times, title, xlabel):
    """
    values_matrix : array [n_cases, n_times]
        e.g. T_mean or F_mean.
    times         : array [n_times]
        e.g. U_times_T or U_times_F.
    """
    n_cases, n_times = values_matrix.shape

    plt.figure(figsize=(8, 5))

    # For a sensible common x-range, use global min/max across all cases & times
    global_min = np.nanmin(values_matrix)
    global_max = np.nanmax(values_matrix)
    x = np.linspace(global_min, global_max, 400)

    for k in range(n_times):
        vals = values_matrix[:, k]
        vals = vals[~np.isnan(vals)]
        if vals.size == 0:
            continue

        mu = np.mean(vals)
        sigma = np.std(vals, ddof=1)

        pdf = _gaussian(x, mu, sigma)
        plt.plot(x, pdf, lw=1.5, label=f"t = {times[k]:.0f} s")

    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Gaussian PDF across cases")
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=7, ncol=3)
    plt.tight_layout()
    plt.show()


# -------- Use for TEMPERATURE (mean across domain) --------
plot_snapshot_gaussians(
    values_matrix=T_mean,        # [n_cases, n_times]
    times=U_times_T,            # same length as second dim of T_mean
    title="Gaussian fits of mean temperature across cases",
    xlabel="Mean temperature T_mean (K)"
)

# You can similarly do this for max/min/variance if desired:
# plot_snapshot_gaussians(T_max, U_times_T, "Gaussian fits of max temperature", "T_max (K)")
# plot_snapshot_gaussians(T_min, U_times_T, "Gaussian fits of min temperature", "T_min (K)")


# -------- Use for LIQUID FRACTION (mean across domain) --------
plot_snapshot_gaussians(
    values_matrix=F_mean,       # [n_cases, n_times]
    times=U_times_F,
    title="Gaussian fits of mean liquid fraction across cases",
    xlabel="Mean liquid fraction f_mean (-)"
)

# Similarly for max/min/variance:
# plot_snapshot_gaussians(F_max, U_times_F, "Gaussian fits of max liquid fraction", "f_max (-)")
# plot_snapshot_gaussians(F_var, U_times_F, "Gaussian fits of Var(f) across cases", "Var(f)")


In [ ]:
# ==========================================
# Gaussian curves across cases for each snapshot (NO Y-AXIS)
# ==========================================

import numpy as np
import matplotlib.pyplot as plt

def _gaussian(x, mu, sigma):
    """Standard normal PDF with mean mu, std sigma."""
    if sigma <= 0:
        sigma = 1e-6
    return (1.0 / (sigma * np.sqrt(2.0 * np.pi))) * np.exp(-0.5 * ((x - mu) / sigma) ** 2)


def plot_snapshot_gaussians(values_matrix, times, title, xlabel):
    """
    values_matrix : array [n_cases, n_times]
    times         : array [n_times]
    Produces a single plot with one Gaussian curve per snapshot time.
    """
    n_cases, n_times = values_matrix.shape

    plt.figure(figsize=(8, 5))

    # Common x-range across all cases for stable plotting
    global_min = np.nanmin(values_matrix)
    global_max = np.nanmax(values_matrix)
    x = np.linspace(global_min, global_max, 400)

    for k in range(n_times):
        vals = values_matrix[:, k]
        vals = vals[~np.isnan(vals)]
        if vals.size == 0:
            continue

        mu = np.mean(vals)
        sigma = np.std(vals, ddof=1)

        pdf = _gaussian(x, mu, sigma)
        plt.plot(x, pdf, lw=1.5, label=f"t = {times[k]:.0f} s")

    plt.title(title)
    plt.xlabel(xlabel)

    # REMOVE Y-AXIS COMPLETELY
    plt.gca().get_yaxis().set_visible(False)

    # REMOVE horizontal grid lines
    plt.grid(False)

    plt.legend(fontsize=7, ncol=3)
    plt.tight_layout()
    plt.show()


# -------- Temperature --------
plot_snapshot_gaussians(
    values_matrix=T_mean,
    times=U_times_T,
    title="Mean temperature across cases",
    xlabel="T_mean (K)"
)

plot_snapshot_gaussians(
    values_matrix=T_max,
    times=U_times_T,
    title="Maximum temperature across cases",
    xlabel="T_max (K)"
)

plot_snapshot_gaussians(
    values_matrix=F_max,
    times=U_times_F,
    title="Maximum liquid fraction across cases",
    xlabel="Max liquid fraction (f_max)"
)
# -------- Liquid Fraction --------
plot_snapshot_gaussians(
    values_matrix=F_mean,
    times=U_times_F,
    title="Mean liquid fraction across cases",
    xlabel="Mean liquid fraction (f_mean)"
)


In [ ]:
# ==========================================
# Separate Gaussian plot for each snapshot
# ==========================================

import numpy as np
import matplotlib.pyplot as plt

def _gaussian(x, mu, sigma):
    """Standard normal PDF."""
    if sigma <= 0:
        sigma = 1e-6
    return (1.0 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - mu) / sigma) ** 2)



def plot_each_snapshot(values_matrix, times, title_prefix, xlabel):
    """
    values_matrix : [n_cases, n_times]
    times         : [n_times]
    Creates separate Gaussian PDF plot for each time snapshot.
    """
    n_cases, n_times = values_matrix.shape

    global_min = np.nanmin(values_matrix)
    global_max = np.nanmax(values_matrix)
    x = np.linspace(global_min, global_max, 400)

    for k in range(n_times):

        vals = values_matrix[:, k]
        vals = vals[~np.isnan(vals)]
        if vals.size == 0:
            continue

        mu = np.mean(vals)
        sigma = np.std(vals, ddof=1)
        pdf = _gaussian(x, mu, sigma)

        plt.figure(figsize=(7,5))
        plt.plot(x, pdf, lw=2, color="navy")
        plt.title(f"{title_prefix} at t = {times[k]:.0f} s")
        plt.xlabel(xlabel)
        plt.ylabel("Gaussian PDF")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()



# ================================
# TEMPERATURE CASES
# ================================
plot_each_snapshot(
    values_matrix=T_mean,
    times=U_times_T,
    title_prefix="Gaussian Curve of Mean Temperature",
    xlabel="Mean Temperature (K)"
)

# Uncomment for max/min/variance as separate snapshots:
# plot_each_snapshot(T_max, U_times_T, "Gaussian Curve of Max Temperature", "T_max (K)")
# plot_each_snapshot(T_min, U_times_T, "Gaussian Curve of Min Temperature", "T_min (K)")
# plot_each_snapshot(T_var, U_times_T, "Gaussian Curve of Temperature Variance", "Var(T)")


# ================================
# LIQUID FRACTION CASES
# ================================
plot_each_snapshot(
    values_matrix=F_mean,
    times=U_times_F,
    title_prefix="Gaussian Curve of Mean Liquid Fraction",
    xlabel="Mean Liquid Fraction (-)"
)

# Uncomment for other metrics:
# plot_each_snapshot(F_max, U_times_F, "Gaussian Curve of Max Liquid Fraction", "f_max")
# plot_each_snapshot(F_min, U_times_F, "Gaussian Curve of Min Liquid Fraction", "f_min")
# plot_each_snapshot(F_var, U_times_F, "Gaussian Curve of Liquid Fraction Variance", "Var(f)")
